# GTAT - GENE TRACE AI

## Cell Line Finder: Integrated MultiOmics Cell Line Selection: Advance Tools for Comprehensive cell line analysis.

### TASK
- Cell line finder and rank the matching cell lines with the confidence level.
- User provides the gene/ gene + protein. With the provided data, need to map the gene and proteins, to map the cell line and rank them with confidence level based on the genes, genomes and transcripts.
- Flag the scores with the gene properties, gene with mutation or fusion gene are flagged.

### DATASETS
#### gene expression
 - 1_4_hpa_rna_celline.tsv
 - 2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.csv
 - 3_GEOexpression.txt
 - 4_Harmonized_MS_CCLE_Gygi_subsetted.csv
#### gene properties
  - 5_OmicsFusionFilteredSupplementary.csv
  - 6_OmicsSomaticMutationsProfile.csv
### nomenclature
  - 7_cellosaurus.csv
  - 8_DepMap_OmicsProfiles.csv
  - 9_DepMap_sample_info.csv
  - 10_GEOInfo.txt
  - 11_hpa_rna_celline_description.tsv
#### non gene expression
  - 12_CCLE_metabolomics_20190502.csv
  - 13_CCLE_miRNA_20181103.gct
  - 14_OmicsGlobalSignatures.csv

# TASK - 1 : Load all the 14 datasets to check the Nature of the datasets


In [ ]:
# Import the necessary packages
import pandas as pd
import numpy as np
import scipy as sp
import re

### HPA DATASET

In [ ]:
# load the hpa dataset from gene expression
hpa_path = "AZ_project_data/data/gene expression/1_4_hpa_rna_celline.tsv"
df_hpa = pd.read_csv(hpa_path, sep='\t')

# print the head
df_hpa.head()

,Gene,Gene name,Cell line,TPM,pTPM,nTPM
0,ENSG00000000003,TSPAN6,143B,22.0,27.6,25.9
1,ENSG00000000003,TSPAN6,22Rv1,2.8,3.6,2.7
2,ENSG00000000003,TSPAN6,23132/87,6.2,7.5,7.5
3,ENSG00000000003,TSPAN6,253J,14.2,18.7,25.4
4,ENSG00000000003,TSPAN6,253J-BV,13.0,17.1,18.5


## HPA Dataset

#### Columns
- Gene : Represent the gene id of the individual genes
- Gene name : Represents the name of the each gene
- Cell line : Represents the cell line the gene belongs
- TPM : Represents the count of the transcripts produced from the gene
- pTPM : Represent the coding genes
- nTPM : Represents the non-coding genes

### DEPMAP DATASET

In [ ]:
# Load the DepMap dataset from gene expression
depmap_path = "AZ_project_data/data/gene expression/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.csv"
df_depmap = pd.read_csv(depmap_path)

# Display the head of the dataset
df_depmap.head()

,Unnamed: 0,TSPAN6 (ENSG00000000003),TNMD (ENSG00000000005),DPM1 (ENSG00000000419),SCYL3 (ENSG00000000457),C1orf112 (ENSG00000000460),FGR (ENSG00000000938),CFH (ENSG00000000971),FUCA2 (ENSG00000001036),GCLC (ENSG00000001084),...,ENSG00000288714,ENSG00000288717,ENSG00000288718,ENSG00000288719,ENSG00000288720,ENSG00000288721,ENSG00000288722,ENSG00000288723,ENSG00000288724,ENSG00000288725
0,PR-AdBjpG,4.331992,0.000000,7.364660,2.792855,4.471187,0.028569,1.226509,3.044394,6.500005,...,0.000000,0.536053,0.000000,0.028569,0.176323,0.992768,2.797013,0.000000,0.0,0.000000
1,PR-I2AzwG,4.567424,0.584963,7.106641,2.543496,3.504620,0.000000,0.189034,3.813525,4.221877,...,0.000000,0.879706,0.000000,0.014355,0.014355,0.432959,2.972693,0.056584,0.0,0.070389
2,PR-5ekAAC,3.150560,0.000000,7.379118,2.333424,4.228049,0.056584,1.310340,6.687201,3.682573,...,0.028569,0.000000,0.084064,0.000000,0.097611,0.367371,1.695994,0.084064,0.0,0.000000
3,PR-I21681,5.085340,0.000000,7.154211,2.545968,3.084064,0.000000,5.868390,6.165309,4.489928,...,0.000000,0.000000,0.070389,0.000000,0.176323,0.411426,3.921246,0.028569,0.0,0.000000
4,PR-i9DRP1,6.729417,0.000000,6.537917,2.456806,3.867896,0.799087,7.208478,5.570159,7.127117,...,0.000000,0.000000,0.201634,0.028569,0.137504,0.678072,4.418190,0.000000,0.0,0.000000


In [ ]:
# Describe the column of the df df_depmap
df_depmap.columns

Index(['Unnamed: 0', 'TSPAN6 (ENSG00000000003)', 'TNMD (ENSG00000000005)',
       'DPM1 (ENSG00000000419)', 'SCYL3 (ENSG00000000457)',
       'C1orf112 (ENSG00000000460)', 'FGR (ENSG00000000938)',
       'CFH (ENSG00000000971)', 'FUCA2 (ENSG00000001036)',
       'GCLC (ENSG00000001084)',
       ...
       'ENSG00000288714', 'ENSG00000288717', 'ENSG00000288718',
       'ENSG00000288719', 'ENSG00000288720', 'ENSG00000288721',
       'ENSG00000288722', 'ENSG00000288723', 'ENSG00000288724',
       'ENSG00000288725'],
      dtype='object', length=53962)

## DepMap

- **Unnamed: 0**: Unique identifier representing the specific cell line or sample (Model ID).
- **Gene Columns (e.g., TSPAN6 (ENSG00000000003))**:
    - **Gene Name/Symbol**: The first part (e.g., TSPAN6).
    - **Ensembl ID**: The ID inside the parentheses (e.g., ENSG00000000003).
    - **Value**: Represents the **Log2(TPM+1)** expression level, which is a normalized measurement of the **TPM (Transcripts Per Million)** for that specific gene in that cell line.

#### NOTE
- Type 1 = gene has a known HGNC symbol → symbol + Ensembl ID shown together
- Type 2 = gene has no assigned symbol yet → only Ensembl ID exists

### GEOexpression

In [ ]:
# Load the GEO expression dataset
geo_expression_path = "AZ_project_data/data/gene expression/3_GEOexpression.txt"
df_geo = pd.read_csv(geo_expression_path, sep='\t')

# Display the head of the dataset
df_geo.head()

,Gene,GSM101610,GSM101615,GSM101616,GSM101667,GSM101668,GSM101671,GSM101672,GSM101673,GSM101674,...,GSM960289,GSM960290,GSM960291,GSM960292,GSM960293,GSM960294,GSM960295,GSM960296,GSM960297,GSM960298
0,ENSG00000000003,33.615700,553.249756,540.452209,599.431152,625.242737,400.554657,412.995605,427.403900,461.123535,...,197.712616,387.427826,116.657890,104.889442,158.400604,736.519043,631.033142,791.054504,181.986893,353.206940
1,ENSG00000000005,40.925682,31.327406,33.934967,34.213123,32.466286,36.243233,37.952511,34.644428,34.225410,...,4.392303,4.230473,4.651096,5.175624,5.540960,4.211264,4.159354,4.480873,4.630062,4.296047
2,ENSG00000000419,2182.281250,3419.430420,3514.540039,2295.817383,2378.469727,2297.564697,2276.069092,2368.761475,2689.160156,...,936.201782,1162.099731,1249.029053,1354.346069,1159.747559,1561.643799,1547.089233,1551.400146,939.964661,1054.350952
3,ENSG00000000457,58.934814,95.068222,94.900459,51.810070,52.327530,50.011375,53.793667,51.172344,52.677490,...,35.778347,57.736485,25.528204,24.593216,23.337006,47.793488,41.268002,64.042831,37.363773,61.027988
4,ENSG00000000460,136.418900,257.929169,271.317230,162.090073,160.913849,206.788635,209.966019,134.272583,147.408554,...,14.693585,19.083935,114.056015,137.948593,149.641708,111.793892,98.536613,167.151016,12.268769,24.296593


In [ ]:
# Col names of the geo expression data
df_geo.columns

Index(['Gene', 'GSM101610', 'GSM101615', 'GSM101616', 'GSM101667', 'GSM101668',
       'GSM101671', 'GSM101672', 'GSM101673', 'GSM101674',
       ...
       'GSM960289', 'GSM960290', 'GSM960291', 'GSM960292', 'GSM960293',
       'GSM960294', 'GSM960295', 'GSM960296', 'GSM960297', 'GSM960298'],
      dtype='object', length=3268)

## GEOexpression

- **Gene**: This column contains the **Ensembl Gene IDs** (e.g., ENSG00000000003), which uniquely identify each gene.
- **GSM Columns (e.g., GSM101610, GSM101615, ...)**:
    - **GSM ID**: These are **GEO Sample Accession numbers**. Each column represents a specific sample or cell line from the Gene Expression Omnibus (GEO).
    - **Values**: These represent the **gene expression levels** (intensity or counts) for that specific gene in that particular sample.

### Harmonized_MS_CCLE_Gygi_subsetted

In [ ]:
# Load the Harmonized MS CCLE Gygi subsetted dataset
gygi_path = "AZ_project_data/data/gene expression/4_Harmonized_MS_CCLE_Gygi_subsetted.csv"
df_gygi = pd.read_csv(gygi_path)

# Display the head of the dataset
df_gygi.head()

,Unnamed: 0,A0AV96 (RBM47),A0AVF1 (IFT56),A0AVG3 (TSNARE1),A0AVI4 (TMEM129),A0AVK6 (E2F8),A0AVT1 (UBA6),A0JLT2 (MED19),A0JNW5 (BLTP3B),A0MZ66 (SHTN1),...,Q9Y6N7-2 (ROBO1),Q9Y4E1-4 (WASHC2C),Q9Y6Q5-2 (AP1M2),Q9Y6K9-2 (IKBKG),Q9Y3Y2-3 (CHTOP),Q9Y4P1-2 (ATG4B),Q9Y6I3-1 (EPN1),Q9Y5V3-2 (MAGED1),Q9Y575-3 (ASB3),Q9Y2L9-2 (LRCH1)
0,ACH-000849,0.358567,-0.172066,NaN,NaN,NaN,-0.496842,0.267691,0.045754,0.065342,...,0.255903,-0.081142,1.680544,0.001122,0.684397,0.397332,0.209930,-0.197547,NaN,-1.100381
1,ACH-000441,-1.112410,0.339446,NaN,NaN,NaN,-0.395633,-0.262715,-0.414909,-0.396370,...,0.199431,-0.300508,-0.414740,0.178375,-0.387377,-0.008456,-0.482339,0.148685,NaN,0.484499
2,ACH-000248,0.855575,-0.181171,NaN,NaN,NaN,-0.284720,-0.436849,0.555351,1.550787,...,-0.849680,0.096941,1.120568,-0.512328,0.102240,-0.204140,0.039421,-0.761104,NaN,-1.191209
3,ACH-000684,0.061377,-0.341233,NaN,NaN,NaN,1.481536,-0.006424,0.185271,-0.125160,...,-0.230147,0.148388,-0.162396,0.279161,-0.057371,-0.331465,-0.014308,0.617058,NaN,0.409230
4,ACH-000856,0.284258,-0.059558,NaN,NaN,NaN,-0.520511,0.680164,-0.281309,-0.691303,...,-0.024407,0.698199,-0.518353,-0.292139,0.498029,0.283140,-0.123974,-0.560028,NaN,-0.120029


In [ ]:
# colnames of the df_gygi
df_gygi.columns

Index(['Unnamed: 0', 'A0AV96 (RBM47)', 'A0AVF1 (IFT56)', 'A0AVG3 (TSNARE1)',
       'A0AVI4 (TMEM129)', 'A0AVK6 (E2F8)', 'A0AVT1 (UBA6)', 'A0JLT2 (MED19)',
       'A0JNW5 (BLTP3B)', 'A0MZ66 (SHTN1)',
       ...
       'Q9Y6N7-2 (ROBO1)', 'Q9Y4E1-4 (WASHC2C)', 'Q9Y6Q5-2 (AP1M2)',
       'Q9Y6K9-2 (IKBKG)', 'Q9Y3Y2-3 (CHTOP)', 'Q9Y4P1-2 (ATG4B)',
       'Q9Y6I3-1 (EPN1)', 'Q9Y5V3-2 (MAGED1)', 'Q9Y575-3 (ASB3)',
       'Q9Y2L9-2 (LRCH1)'],
      dtype='object', length=12559)

## Harmonized MS CCLE Gygi (Mass Spectrometry Proteomics)

- **Unnamed: 0**: This column contains the **DepMap Model ID** (e.g., ACH-000849), which uniquely identifies each cell line.
- **Protein Columns (e.g., A0AV96 (RBM47))**:
    - **Uniprot ID**: The first part of the header (e.g., A0AV96) refers to the unique protein identifier in the UniProt database.
    - **Gene Symbol**: The symbol in parentheses (e.g., RBM47) is the gene associated with that protein.
    - **Values**: These represent **relative protein expression levels** (log-transformed ratios) measured via mass spectrometry.

#### Note
Unlike the previous transcriptomic datasets (RNA-seq), this dataset measures the actual **protein levels**, providing a closer look at the functional state of the cell lines.

### Omic Fusion Fitered Supplementry

In [ ]:
# Load the Omics Fusion Filtered Supplementary dataset
fusion_path = "AZ_project_data/data/gene properties/5_OmicsFusionFilteredSupplementary.csv"
df_fusion = pd.read_csv(fusion_path)

# Display the head of the dataset
df_fusion.head()

,Unnamed: 0,SequencingID,ModelID,IsDefaultEntryForModel,ModelConditionID,IsDefaultEntryForMC,CanonicalFusionName,gene1(ENS ID),gene2(ENS ID),TotalReadsInSample,...,breakpoint2,site1,site2,type,coverage1,coverage2,tags,retained_protein_domains,direction1,direction2
0,0,CDS-010xbm,ACH-001113,Yes,MC-001113-k2lR,Yes,DLG1--SERPINI1,DLG1 (ENSG00000075711.21),SERPINI1 (.),47434547,...,chr3:167836645,CDS/splice-site,intergenic,inversion,1454,76,.,.,upstream,upstream
1,1,CDS-010xbm,ACH-001113,Yes,MC-001113-k2lR,Yes,ADAM17--ITGB1BP1,ADAM17 (ENSG00000151694.14),ITGB1BP1 (ENSG00000119185.13),47434547,...,chr2:9414256,CDS/splice-site,CDS/splice-site,deletion/read-through,883,724,.,.,upstream,downstream
2,2,CDS-010xbm,ACH-001113,Yes,MC-001113-k2lR,Yes,ADAM17--ITGB1BP1,ADAM17 (ENSG00000151694.14),ITGB1BP1 (ENSG00000119185.13),47434547,...,chr2:9412405,CDS/splice-site,CDS/splice-site,deletion/read-through,883,754,.,.,upstream,downstream
3,3,CDS-010xbm,ACH-001113,Yes,MC-001113-k2lR,Yes,ADAM17--ITGB1BP1,ADAM17 (ENSG00000151694.14),ITGB1BP1 (ENSG00000119185.13),47434547,...,chr2:9414256,exon/splice-site,CDS/splice-site,deletion/read-through,41,724,.,.,upstream,downstream
4,4,CDS-010xbm,ACH-001113,Yes,MC-001113-k2lR,Yes,ADAM17--ITGB1BP1,ADAM17 (ENSG00000151694.14),ITGB1BP1 (ENSG00000119185.13),47434547,...,chr2:9408205,CDS/splice-site,CDS/splice-site,deletion/read-through,883,680,.,.,upstream,downstream


In [ ]:
# col names of df_fusion
df_fusion.columns

Index(['Unnamed: 0', 'SequencingID', 'ModelID', 'IsDefaultEntryForModel',
       'ModelConditionID', 'IsDefaultEntryForMC', 'CanonicalFusionName',
       'gene1(ENS ID)', 'gene2(ENS ID)', 'TotalReadsInSample',
       'TotalReadsSupportingFusion', 'FFPM', 'confidence', 'split_reads1',
       'split_reads2', 'discordant_mates', 'strand1(gene/fusion)',
       'strand2(gene/fusion)', 'reading_frame', 'breakpoint1', 'breakpoint2',
       'site1', 'site2', 'type', 'coverage1', 'coverage2', 'tags',
       'retained_protein_domains', 'direction1', 'direction2'],
      dtype='object')

## Omics Fusion Filtered Supplementary

This dataset describes **gene fusions**, which occur when two previously separate genes become joined together, often seen in cancer.

- **ModelID**: The DepMap identifier for the cell line (e.g., ACH-001113).
- **CanonicalFusionName**: The names of the two genes involved in the fusion (e.g., DLG1--SERPINI1).
- **gene1(ENS ID) / gene2(ENS ID)**: The specific Ensembl IDs for the upstream and downstream genes involved.
- **type**: The biological mechanism of the fusion (e.g., inversion, deletion/read-through, translocation).
- **breakpoint1 / breakpoint2**: The exact genomic coordinates where the DNA break and fusion occurred.
- **site1 / site2**: Describes where the fusion happens relative to gene structures (e.g., CDS/splice-site, intergenic).
- **coverage1 / coverage2**: Numerical values indicating the sequencing depth or evidence support for the fusion event at that specific site.

#### Significance
Fusion genes can act as powerful oncogenes. Identifying these in specific cell lines helps in understanding the genomic drivers of that specific model.

### Omic Somantic Mutation Profile

In [ ]:
# Load the Omics Somatic Mutations Profile dataset
mutation_path = "AZ_project_data/data/gene properties/6_OmicsSomaticMutationsProfile.csv"
df_mutation = pd.read_csv(mutation_path)

# Display the head of the dataset
df_mutation.head()

C:\Users\Thiruvel A P\AppData\Local\Temp\ipykernel_46028\1750849819.py:3: DtypeWarning: Columns (22,50,54,56,57,58,59,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mutation = pd.read_csv(mutation_path)


,Chrom,Pos,Ref,Alt,AF,DP,RefCount,AltCount,GT,PS,...,GwasPmID,GtexGene,ProveanPrediction,AMClass,AMPathogenicity,Rescue,RescueReason,ProfileID,Hotspot,EntrezGeneID
0,chr1,818203,G,A,0.240,27,21,6,0/1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,PR-t8SaQo,False,400728.0
1,chr1,851926,G,A,0.158,17,15,2,0/1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,PR-lhMBt6,False,643837.0
2,chr1,924510,GC,AA,0.412,35,21,14,0|1,924510.0,...,NaN,NaN,NaN,NaN,NaN,False,NaN,PR-uKiczK,False,148398.0
3,chr1,924657,C,G,0.437,17,9,8,0/1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,PR-sxFiuq,False,148398.0
4,chr1,924750,C,T,0.625,19,7,12,0/1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,PR-DNEoiz,False,148398.0


In [ ]:
# Col names of the df_mutation
df_mutation.columns

Index(['Chrom', 'Pos', 'Ref', 'Alt', 'AF', 'DP', 'RefCount', 'AltCount', 'GT',
       'PS', 'VariantType', 'VariantInfo', 'DNAChange', 'ProteinChange',
       'HugoSymbol', 'Exon', 'Intron', 'EnsemblGeneID', 'EnsemblFeatureID',
       'HgncName', 'HgncFamily', 'UniprotID', 'DbsnpRsID', 'GcContent',
       'LofGeneName', 'LofGeneId', 'LofNumberOfTranscriptsInGene',
       'LofPercentOfTranscriptsAffected', 'NMD', 'MolecularConsequence',
       'VepImpact', 'VepBiotype', 'VepHgncID', 'VepExistingVariation',
       'VepManeSelect', 'VepENSP', 'VepSwissprot', 'Sift', 'Polyphen',
       'GnomadeAF', 'GnomadgAF', 'VepClinSig', 'VepSomatic', 'VepPliGeneValue',
       'VepLofTool', 'OncogeneHighImpact', 'TumorSuppressorHighImpact',
       'TranscriptLikelyLof', 'Brca1FuncScore', 'CivicID', 'CivicDescription',
       'CivicScore', 'LikelyLoF', 'HessDriver', 'HessSignature', 'RevelScore',
       'PharmgkbId', 'DidaID', 'DidaName', 'GwasDisease', 'GwasPmID',
       'GtexGene', 'ProveanPrediction'

## Omics Somatic Mutation Profile

This dataset contains detailed information about **somatic mutations** genetic alterations that occur in non-reproductive cells. These are key drivers in cancer development.

- **Chrom / Pos**: The genomic location (Chromosome and Position) of the mutation.
- **Ref / Alt**: The **Reference** allele (normal) vs the **Alternative** allele (mutated version).
- **AF (Allele Frequency)**: The proportion of DNA sequences in the sample that carry the mutation.
- **ProteinChange**: Describes how the mutation affects the resulting protein (e.g., p.V600E).
- **VariantInfo**: Categorizes the mutation type (e.g., Missense, Nonsense, Frame_Shift_Del).
- **Hotspot**: A boolean flag (True/False) indicating if the mutation occurs at a known 'hotspot' frequently mutated in cancer.
- **ProfileID**: Links the mutation record to a specific DepMap profile.

#### Significance
These mutations serve as 'flags'. If a user is interested in a specific gene, knowing if it is mutated in a particular cell line is crucial for selecting the right biological model for experiments.

### Geo Info

In [ ]:
# Load the GEO Info dataset from nomenclature
geo_info_path = "AZ_project_data/data/nomenclature/10_GEOInfo.txt"
df_geo_info = pd.read_csv(geo_info_path, sep='\t')

# Display the head of the dataset
df_geo_info.head()

,Geo_accession,CEL_file_names,title,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,contact_institute,GSE_ID,GSE_filename,cell_line,disease,origin,Cellosaurus_ID,Cellline,Matching_Type,cell_line_Trimmed
0,GSM101610,GSM101610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_0131,A-172,Cello GEO GSM,NaN
1,GSM101615,GSM101615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_0393,LN-229,Cello GEO GSM,NaN
2,GSM101616,GSM101616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_0393,LN-229,Cello GEO GSM,NaN
3,GSM101667,GSM101667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_1715,SW1088,Cello GEO GSM,NaN
4,GSM101668,GSM101668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_1715,SW1088,Cello GEO GSM,NaN


## GEO Info

This dataset acts as a bridge between the **GEO expression data** and biological metadata, allowing us to identify which samples correspond to which cell lines.

- **Geo_accession**: The unique identifier for each sample (matches the columns in `df_geo`).
- **Cellline**: The common name of the biological cell line (e.g., A-172, LN-229).
- **Cellosaurus_ID**: The unique identifier from the Cellosaurus database, providing a standardized way to reference cell lines.
- **disease**: Information regarding the disease state of the sample (if available).
- **origin**: The tissue or biological origin of the cell line.

#### Significance
This is a critical lookup table. Without it, the `GSM` IDs in the expression dataset are just anonymous identifiers. This mapping allows us to relate gene expression values back to specific cancer models or cell types.

### HPA RNA Cell line

In [ ]:
# Load the HPA RNA cell line description dataset
hpa_desc_path = "AZ_project_data/data/nomenclature/11_hpa_rna_celline_description.tsv"
df_hpa_desc = pd.read_csv(hpa_desc_path, sep='\t')

# Display the head of the dataset
df_hpa_desc.head()

,Cell line,Disease,Disease subtype,Cellosaurus ID,Patient,Primary/Metastasis,Sample collection site
0,143B,Bone cancer,Osteosarcoma,CVCL_2270,13,Primary,bone
1,22Rv1,Prostate cancer,Adenocarcinoma,CVCL_1045,Male,primary,prostate
2,23132/87,Gastric cancer,Adenocarcinoma,CVCL_1046,"Male, 72",primary,stomach
3,253J,Bladder cancer,Carcinoma,CVCL_7935,"Male, 53",metastasis,lymph node
4,253J-BV,Bladder cancer,Carcinoma,CVCL_7937,"Male, 53",metastasis,lymph node


## HPA RNA Cell Line Description

This dataset contains metadata for the cell lines featured in the Human Protein Atlas (HPA) transcriptomic data.

- **Cell line**: The standard name of the cell line (matches the 'Cell line' column in `df_hpa`).
- **Disease**: The broad category of cancer or condition associated with the cell line (e.g., Bone cancer, Prostate cancer).
- **Disease subtype**: More specific histological classification (e.g., Osteosarcoma, Adenocarcinoma).
- **Cellosaurus ID**: The unique identifier for the cell line in the Cellosaurus database.
- **Patient**: Demographic information about the donor (e.g., age and sex).
- **Primary/Metastasis**: Indicates whether the sample was taken from the primary tumor site or a metastatic site.
- **Sample collection site**: The specific tissue or organ where the sample was harvested.

#### Significance
This dataset allows us to filter or group gene expression results from the HPA based on biological criteria like disease type or metastatic status, which is essential for identifying cancer-specific expression patterns.

### Cellosaurus


In [ ]:
# Load the Cellosaurus dataset from nomenclature
cellosaurus_path = "AZ_project_data/data/nomenclature/7_cellosaurus.csv"
df_cellosaurus = pd.read_csv(cellosaurus_path)

# Display the head of the dataset
df_cellosaurus.head()

,Identifier (cell line name),Accession (CVCL_xxxx),Secondary accession number(s),Synonyms,Cross-references,References identifiers,Web pages,Comments,STR profile data,Diseases,Species of origin,Hierarchy,Originate from same individual,Sex of cell,Age of donor at sampling,Category,Date (entry history)
0,#132 PC3-1-SC-E8,CVCL_B0T9,NaN,Z48-5MG-70,Wikidata; Q108819335,Patent=EP0501779A1;,NaN,Group: Patented cell line. || Registration: In...,NaN,NaN,NCBI_TaxID=10090; ! Mus musculus (Mouse),CVCL_D145 ! HL-1 Friendly Myeloma-653,NaN,NaN,NaN,Hybridoma,Created: 23-09-21; Last updated: 30-01-24; Ver...
1,#132 PL12 SC-D1,CVCL_B0T8,NaN,Z48-5MG-63,Wikidata; Q108819336,Patent=EP0501779A1;,NaN,Group: Patented cell line. || Registration: In...,NaN,NaN,NCBI_TaxID=10090; ! Mus musculus (Mouse),CVCL_D145 ! HL-1 Friendly Myeloma-653,NaN,NaN,NaN,Hybridoma,Created: 23-09-21; Last updated: 30-01-24; Ver...
2,#15310-LN,CVCL_E548,NaN,15310-LN; TER461; TER-461; Ter 461; TER479; TE...,dbMHC; 48439 || ECACC; 94050311 || IHW; IHW093...,NaN,http://pathology.ucla.edu/workfiles/360cx.pdf ...,Part of: 12th International Histocompatibility...,NaN,NaN,NCBI_TaxID=9606; ! Homo sapiens (Human),NaN,NaN,Female,Age unspecified,Transformed cell line,Created: 22-10-12; Last updated: 30-01-24; Ver...
3,#16-15,CVCL_KA96,NaN,NaN,RCB; RCB4635 || Wikidata; Q54422067,PubMed=25400923;,NaN,Monoclonal antibody isotype: IgM. || Monoclona...,NaN,NaN,NCBI_TaxID=10090; ! Mus musculus (Mouse) || NC...,CVCL_4032 ! P3X63Ag8.653,NaN,NaN,NaN,Hybridoma,Created: 22-08-17; Last updated: 21-03-23; Ver...
4,#40a,CVCL_IW91,NaN,NaN,Wikidata; Q54422071,PubMed=28159921;,NaN,Characteristics: Established from parent cell ...,NaN,NCIt; C21619; Mouse mesothelioma,NCBI_TaxID=10090; ! Mus musculus (Mouse),CVCL_IW90 ! 40,NaN,Male,1-2M,Cancer cell line,Created: 15-05-17; Last updated: 29-06-23; Ver...


In [ ]:
# column names of the df
df_cellosaurus.columns

Index(['Identifier (cell line name)', 'Accession (CVCL_xxxx)',
       'Secondary accession number(s)', 'Synonyms', 'Cross-references',
       'References identifiers', 'Web pages', 'Comments', 'STR profile data',
       'Diseases', 'Species of origin', 'Hierarchy',
       'Originate from same individual', 'Sex of cell',
       'Age of donor at sampling', 'Category', 'Date (entry history)'],
      dtype='object')

## Cellosaurus

The Cellosaurus dataset is a master resource for cell line nomenclature and metadata, providing a cross-reference for all experimental models used in the other datasets.

- **Identifier (cell line name)**: The primary common name used for the cell line.
- **Accession (CVCL_xxxx)**: The unique and permanent identifier in the Cellosaurus database.
- **Synonyms**: Alternative names that the cell line might be known by across different labs or publications.
- **Species of origin**: The organism from which the cell line was derived (e.g., *Homo sapiens*, *Mus musculus*).
- **Diseases**: The specific medical condition or cancer type associated with the cell line origin.
- **Sex of cell / Age of donor**: Demographic details of the biological source.
- **Category**: The type of cell line (e.g., Cancer cell line, Hybridoma, Transformed cell line).

#### Significance
This dataset is the "source of truth" for resolving naming conflicts. Since different datasets (like GEO or DepMap) might use slightly different names for the same cell line, the **CVCL Accession** allows us to link them all together accurately.

### DepMap Omics

In [ ]:
# Load the DepMap Omics Profiles dataset from nomenclature
depmap_omics_path = "AZ_project_data/data/nomenclature/8_DepMap_OmicsProfiles.csv"
df_depmap_omics = pd.read_csv(depmap_omics_path)

# Display the head of the dataset
df_depmap_omics.head()

,ProfileID,ModelCondition,ModelID,Datatype,WESKit
0,PR-00UtU3,MC-001131-kkJv,ACH-001131,wgs,NaN
1,PR-01r7OM,MC-000957-Yckn,ACH-000957,rna,NaN
2,PR-02XmLG,MC-002785-qo9e,ACH-002785,rna,NaN
3,PR-04VvBz,MC-001289-BpdI,ACH-001289,wes,ICE
4,PR-09gmEI,MC-000520-YIm7,ACH-000520,rna,NaN


## DepMap Omics Profiles

This dataset provides a mapping between specific experimental profiles (e.g., a specific RNA-seq run) and the stable Model IDs for the cell lines.

- **ProfileID**: A unique identifier for a specific sequencing or data collection event (e.g., `PR-01r7OM`). This ID is used as the column header in the `df_depmap` (DepMap Expression) dataset.
- **ModelID**: The stable identifier for the cell line (e.g., `ACH-000957`). This allows us to link expression profiles back to cell line metadata in `DepMap_sample_info`.
- **ModelCondition**: Describes the specific state of the model during profiling.
- **Datatype**: Indicates the type of omics data (e.g., `rna` for RNA-seq, `wgs` for Whole Genome Sequencing, `wes` for Whole Exome Sequencing).

#### Significance
This is the "Rosetta Stone" for DepMap data. Because a single cell line (ModelID) might have multiple sequencing profiles (ProfileIDs), this table allows us to correctly associate the large expression matrices with the underlying biological models.

### DepMap Samples

In [ ]:
# Load the DepMap Sample Info dataset from nomenclature
depmap_sample_path = "AZ_project_data/data/nomenclature/9_DepMap_sample_info.csv"
df_depmap_sample = pd.read_csv(depmap_sample_path)

# Display the head of the dataset
df_depmap_sample.head()

,DepMap_ID,cell_line_name,stripped_cell_line_name,CCLE_Name,alias,COSMICID,sex,source,RRID,WTSI_Master_Cell_ID,...,lineage_sub_subtype,lineage_molecular_subtype,default_growth_pattern,model_manipulation,model_manipulation_details,patient_id,parent_depmap_id,Cellosaurus_NCIt_disease,Cellosaurus_NCIt_id,Cellosaurus_issues
0,ACH-000016,SLR 21,SLR21,SLR21_KIDNEY,NaN,NaN,NaN,Academic lab,CVCL_V607,NaN,...,NaN,NaN,NaN,NaN,NaN,PT-JnARLB,NaN,Clear cell renal cell carcinoma,C4033,NaN
1,ACH-000032,MHH-CALL-3,MHHCALL3,MHHCALL3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,NaN,Female,DSMZ,CVCL_0089,NaN,...,b_cell,NaN,NaN,NaN,NaN,PT-p2KOyI,NaN,Childhood B acute lymphoblastic leukemia,C9140,NaN
2,ACH-000033,NCI-H1819,NCIH1819,NCIH1819_LUNG,NaN,NaN,Female,Academic lab,CVCL_1497,NaN,...,NSCLC_adenocarcinoma,NaN,NaN,NaN,NaN,PT-9p1WQv,NaN,Lung adenocarcinoma,C3512,NaN
3,ACH-000043,Hs 895.T,HS895T,HS895T_FIBROBLAST,NaN,NaN,Female,ATCC,CVCL_0993,NaN,...,NaN,NaN,2D: adherent,NaN,NaN,PT-rTUVZQ,NaN,Melanoma,C3224,NaN
4,ACH-000049,HEK TE,HEKTE,HEKTE_KIDNEY,NaN,NaN,NaN,Academic lab,CVCL_WS59,NaN,...,NaN,NaN,NaN,immortalized,NaN,PT-qWYYgr,NaN,NaN,NaN,No information is available about this cell li...


## DepMap Sample Info

This dataset provides the comprehensive metadata for every cell line model included in the DepMap project.

- **DepMap_ID**: The unique and stable identifier (e.g., `ACH-000016`) for the model. This is used to link to the `ModelID` in the Omics Profiles.
- **cell_line_name**: The standard common name for the cell line.
- **lineage / lineage_subtype**: Categorizes the cell line by tissue of origin and specific histological type.
- **COSMICID / RRID**: External identifiers that link these models to other major biological databases.
- **Cellosaurus_NCIt_disease**: The standardized disease name using the NCI Thesaurus terminology.
- **sex / age**: Demographic information about the patient from whom the cell line was derived.

#### Significance
This file is the primary source for biological context. When we find an interesting expression or mutation pattern in a specific DepMap ID, we use this table to determine what kind of cancer it represents and its tissue of origin.

### CCLE_metabolomics_20190502

In [ ]:
# Load the CCLE metabolomics dataset
metabolomics_path = "AZ_project_data/data/non gene expression/12_CCLE_metabolomics_20190502.csv"
df_metabolomics = pd.read_csv(metabolomics_path)

# Display the head of the dataset
display(df_metabolomics.head())

,CCLE_ID,DepMap_ID,2-aminoadipate,3-phosphoglycerate,alpha-glycerophosphate,4-pyridoxate,aconitate,adenine,adipate,alpha-ketoglutarate,...,C56:8 TAG,C56:7 TAG,C56:6 TAG,C56:5 TAG,C56:4 TAG,C56:3 TAG,C56:2 TAG,C58:8 TAG,C58:7 TAG,C58:6 TAG
0,DMS53_LUNG,ACH-000698,6.112727,6.034198,5.896896,6.000532,5.513618,5.868529,5.977177,5.693074,...,6.070239,6.133433,6.091089,6.257711,6.372732,6.202511,5.939576,6.309821,6.115974,5.999436
1,SW1116_LARGE_INTESTINE,ACH-000489,5.577413,5.727045,5.111468,6.073250,5.802494,5.824473,5.888821,5.768379,...,6.248653,6.633575,6.378052,6.341043,6.360945,6.333540,6.137271,7.065858,6.832174,6.363064
2,NCIH1694_LUNG,ACH-000431,5.886398,5.574881,5.541259,5.848375,5.665026,5.875548,5.894904,5.839640,...,5.942887,5.946988,5.837980,5.913350,6.137530,5.807546,5.704149,5.881193,5.785208,5.504225
3,P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,ACH-000707,5.770030,6.099229,6.233259,5.543495,5.767759,6.155905,6.111148,5.949481,...,6.516922,6.113791,6.282113,6.248667,6.109480,6.043570,5.846802,6.429402,5.779815,6.241530
4,HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,ACH-000509,5.480683,5.469742,6.509397,6.251005,5.190578,5.897085,6.148333,5.607481,...,6.161981,6.777932,6.676390,6.695659,6.751029,6.385056,6.682612,6.757899,6.728570,6.879260


In [ ]:
# col names of the df
df_metabolomics.columns

Index(['CCLE_ID', 'DepMap_ID', '2-aminoadipate', '3-phosphoglycerate',
       'alpha-glycerophosphate', '4-pyridoxate', 'aconitate', 'adenine',
       'adipate', 'alpha-ketoglutarate',
       ...
       'C56:8 TAG', 'C56:7 TAG', 'C56:6 TAG', 'C56:5 TAG', 'C56:4 TAG',
       'C56:3 TAG', 'C56:2 TAG', 'C58:8 TAG', 'C58:7 TAG', 'C58:6 TAG'],
      dtype='object', length=227)

## CCLE Metabolomics

The df_metabolomics dataset contains 227 columns. Here is a detailed breakdown of the column types and their roles:
- CCLE_ID: This is a human-readable identifier for the cell line, typically following the format [CellLineName]_[Tissue]. For example, DMS53_LUNG.
- DepMap_ID: This is the primary key used across most DepMap/CCLE datasets (e.g., ACH-000698). It is the most reliable ID for joining this data with the expression or mutation datasets we loaded earlier.
Metabolite Columns (e.g., 2-aminoadipate, 3-phosphoglycerate, alpha-glycerophosphate): The remaining 225 columns represent specific metabolites.
- Small Molecules: Common metabolic intermediates like amino acids, organic acids, and sugars.
Lipids: Many columns at the end of the dataframe (like C56:8 TAG) refer to Triacylglycerols (TAGs), where the numbers indicate the carbon chain length and number of double bonds.
- Values: The numerical values in these columns represent the relative abundance of each metabolite. These are usually log-transformed to stabilize variance and allow for comparison across different metabolites.


This dataset allows us to see the metabolic "fingerprint" of each cancer cell line, which can be linked back to its genetic mutations or gene expression levels.

### CCLE miRNA

In [ ]:
# The file is in GCT format, which often requires skipping metadata lines or specific parsing
# We will attempt to load the CCLE miRNA data
mirna_path = "AZ_project_data/data/non gene expression/13_CCLE_miRNA_20181103.gct"

# GCT files typically have 2 lines of metadata before the header
df_mirna = pd.read_csv(mirna_path, sep='\t', skiprows=2)

# Display the head of the dataset
display(df_mirna.head())

,Name,Description,DMS53_LUNG,SW1116_LARGE_INTESTINE,NCIH1694_LUNG,P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,UMUC3_URINARY_TRACT,HOS_BONE,HUNS1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,...,MOLT3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,HOP62_LUNG,EKVX_LUNG,OVCAR5_OVARY,UO31_KIDNEY,SF268_CENTRAL_NERVOUS_SYSTEM,SF539_CENTRAL_NERVOUS_SYSTEM,SNB75_CENTRAL_NERVOUS_SYSTEM,HOP92_LUNG,MUTZ3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
0,nmiR00001.1,hsa-let-7a,4362.58,5191.50,24991.05,3253.83,225.28,35051.17,5706.63,1821.46,...,30327.89,32129.83,18651.43,17551.39,30510.57,23953.54,26114.02,14115.97,13986.26,244.46
1,nmiR00002.1,hsa-let-7b,187.44,868.22,5066.09,74.21,35.74,1014.65,1211.27,22.54,...,2618.06,7962.03,3990.50,2629.28,5482.37,2718.48,2094.48,1318.81,1893.46,98.57
2,nmiR00003.1,hsa-let-7c,267.03,244.88,818.49,53.28,27.19,473.15,163.82,437.44,...,431.15,1573.10,528.74,506.52,1019.14,491.97,424.39,1830.06,355.16,77.54
3,nmiR00004.1,hsa-let-7d,868.11,556.55,3661.86,315.87,94.00,1766.54,904.84,133.18,...,8721.99,6134.93,6059.74,4448.79,4296.90,4183.58,6098.12,3645.09,970.48,197.15
4,nmiR00005.1,hsa-let-7e,1.04,92.02,942.62,1.90,0.77,279.63,168.07,1.02,...,99.07,2184.77,3893.23,806.47,1455.61,508.19,1872.56,1188.79,685.53,1.31


In [ ]:
# col names in df_mirna
df_mirna.columns

Index(['Name', 'Description', 'DMS53_LUNG', 'SW1116_LARGE_INTESTINE',
       'NCIH1694_LUNG', 'P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE',
       'HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE', 'UMUC3_URINARY_TRACT',
       'HOS_BONE', 'HUNS1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE',
       ...
       'MOLT3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE', 'HOP62_LUNG', 'EKVX_LUNG',
       'OVCAR5_OVARY', 'UO31_KIDNEY', 'SF268_CENTRAL_NERVOUS_SYSTEM',
       'SF539_CENTRAL_NERVOUS_SYSTEM', 'SNB75_CENTRAL_NERVOUS_SYSTEM',
       'HOP92_LUNG', 'MUTZ3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'],
      dtype='object', length=956)

### CCLE miRNA

The CCLE miRNA dataset (df_mirna) is structured as an expression matrix with 956 columns. Here is a breakdown of what those columns represent:

- Name: This column contains internal identifiers for the miRNAs (e.g., nmiR00001.1). These are unique keys used within the CCLE processing pipeline to identify specific microRNA sequences.
- Description: This column provides the standardized biological names for the miRNAs using miRBase nomenclature (e.g., hsa-let-7a). This is usually the primary column used for biological interpretation, as it tells you exactly which microRNA is being measured.
- Cell Line Columns (e.g., DMS53_LUNG, SW1116_LARGE_INTESTINE, NCIH1694_LUNG): The remaining 954 columns represent individual cancer cell lines.
- Naming Convention: The headers use the [CellLineName]_[Tissue] format, which matches the CCLE_ID found in the metabolomics and sample info datasets.
- Values: The numerical values represent the relative expression levels of each miRNA in that specific cell line. These values indicate how much of a particular miRNA is present, allowing for comparison across different cancer types.


This dataset is particularly valuable because miRNAs are key regulators of gene expression, and their levels can often explain why certain genes are suppressed in specific cell lines.

### OmicsGlobalSignatures

In [ ]:
# Load the Omics Global Signatures dataset
global_signatures_path = "AZ_project_data/data/non gene expression/14_OmicsGlobalSignatures.csv"
df_global_signatures = pd.read_csv(global_signatures_path)

# Display the head of the dataset
display(df_global_signatures.head())

,Unnamed: 0,SequencingID,ModelID,ModelConditionID,IsDefaultEntryForModel,IsDefaultEntryForMC,MSIScore,LoHFraction,WGD,CIN,Ploidy,Aneuploidy
0,0,CDS-00Nrci,ACH-000839,MC-000839-krru,Yes,Yes,3.68,0.107443,1.0,0.502634,3.158291,20.0
1,1,CDS-051xn7,ACH-000041,MC-000041-uPBf,Yes,Yes,2.21,0.130089,1.0,0.523865,3.236089,19.0
2,2,CDS-099jzP,ACH-002046,MC-002046-oaX8,Yes,Yes,2.87,0.222342,1.0,0.679772,3.326715,30.0
3,3,CDS-0A4mDu,ACH-002048,MC-002048-52d6,Yes,Yes,2.48,NaN,NaN,NaN,NaN,NaN
4,4,CDS-0Eax8o,ACH-000042,MC-000042-eOnX,Yes,Yes,2.07,NaN,NaN,NaN,NaN,NaN


In [ ]:
# col names for the df_glabal_signature
df_global_signatures.columns

Index(['Unnamed: 0', 'SequencingID', 'ModelID', 'ModelConditionID',
       'IsDefaultEntryForModel', 'IsDefaultEntryForMC', 'MSIScore',
       'LoHFraction', 'WGD', 'CIN', 'Ploidy', 'Aneuploidy'],
      dtype='object')

## Omics Global Signatures

The df_global_signatures dataset provides high-level genomic metrics that describe the overall state of a cell line's genome. Here is a breakdown of the columns:

#### Identifiers and Metadata
- Unnamed: 0: A simple row index.
- SequencingID: A unique identifier for the specific sequencing run or data processing batch (e.g., CDS-00Nrci).
- ModelID: The stable DepMap identifier for the cell line (e.g., ACH-000839). This is the key used to link this data to other DepMap datasets.
- ModelConditionID: Identifies the specific biological condition of the model when it was sequenced.
- IsDefaultEntryForModel / IsDefaultEntryForMC: Boolean flags indicating if this specific sequencing record is the primary (default) representation for that cell line or condition.

#### Genomic Signatures
- MSIScore (Microsatellite Instability): A numerical score indicating the level of microsatellite instability. High MSI is often caused by defects in DNA mismatch repair.
- LoHFraction (Loss of Heterozygosity): Measures the fraction of the genome that has lost one of the two parental alleles.
- WGD (Whole Genome Doubling): Indicates whether the cell line has undergone a whole-genome doubling event (often 0 for no, 1 for yes).
- CIN (Chromosomal Instability): A score representing the rate of numerical and structural chromosomal changes.
- Ploidy: The average number of sets of chromosomes in the cell (e.g., 2.0 is diploid, higher values indicate polyploidy).
- Aneuploidy: A score indicating the degree of deviation from an exact multiple of the haploid number of chromosomes (e.g., gain or loss of specific chromosomes).
- These columns allow researchers to categorize cell lines by their global genomic landscape, which often correlates with how they respond to different treatments.

# BASIC EDA

In [ ]:
def perform_comprehensive_eda(df, df_name="Dataset"):
    """
    Performs generic EDA on a dataframe and prints a summary for each column.
    """
    print(f"{'='*30} EDA: {df_name} {'='*30}")
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

    # Global stats
    duplicates_count = df.duplicated().sum()
    print(f"Total Duplicate Rows: {duplicates_count}")

    # Column-wise analysis
    eda_results = []

    for col in df.columns:
        col_data = df[col]
        dtype = col_data.dtype
        missing = col_data.isnull().sum()
        unique_count = col_data.nunique()
        is_unique_id = "Yes" if unique_count == len(df) else "No"

        # Initialize column info
        info = {
            "Column": col,
            "Dtype": dtype,
            "Missing": missing,
            "Unique": unique_count,
            "Is_UID": is_unique_id
        }

        # Numerical Analysis
        if np.issubdtype(dtype, np.number):
            info["Mean"] = round(col_data.mean(), 2)
            info["Std"] = round(col_data.std(), 2)
            # Scaling check (Min/Max range)
            info["Range"] = f"[{col_data.min()}, {col_data.max()}]"

            # Outlier detection using IQR
            q1 = col_data.quantile(0.25)
            q3 = col_data.quantile(0.75)
            iqr = q3 - q1
            outliers = col_data[(col_data < (q1 - 1.5 * iqr)) | (col_data > (q3 + 1.5 * iqr))].count()
            info["Outliers"] = outliers
            info["Non-AlphaNum"] = "N/A"

        # Categorical/String Analysis
        else:
            # Presence of non-alphanumeric values (excluding spaces/underscores common in biological data)
            # We check a sample to maintain performance on huge tables
            sample_vals = col_data.dropna().astype(str).head(1000)
            has_special = any(re.search(r'[^a-zA-Z0-9\s_\-]', val) for val in sample_vals)
            info["Non-AlphaNum"] = "Yes" if has_special else "No"
            info["Mean/Std/Outliers"] = "N/A"

        eda_results.append(info)

    # Print summary as a DataFrame for readability
    summary_df = pd.DataFrame(eda_results)
    # If the dataframe is huge (like DepMap), show head/tail
    if len(summary_df) > 20:
        display(summary_df.head(10))
        print("...")
        display(summary_df.tail(10))
    else:
        display(summary_df)

    # Basic Correlation for numerical columns (limited to first 20 numeric columns for performance)
    numeric_df = df.select_dtypes(include=[np.number])
    if not numeric_df.empty and numeric_df.shape[1] > 1:
        print("\n--- Correlation (Top Numerical Columns) ---")
        corr_cols = numeric_df.columns[:20]
        display(numeric_df[corr_cols].corr().round(2))

### 1. HPA Dataset

In [ ]:
# Call the generic EDA function for the HPA dataset
perform_comprehensive_eda(df_hpa, df_name="HPA RNA Cell Line")

============================== EDA: HPA RNA Cell Line ==============================
Shape: 24315372 rows, 6 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
0,Gene,object,0,20162,No,No,N/A,NaN,NaN,NaN,NaN
1,Gene name,object,0,20151,No,No,N/A,NaN,NaN,NaN,NaN
2,Cell line,object,0,1206,No,Yes,N/A,NaN,NaN,NaN,NaN
3,TPM,float64,0,55888,No,N/A,NaN,38.94,384.10,"[0.0, 153461.5]",2900058.0
4,pTPM,float64,0,64603,No,N/A,NaN,49.56,489.40,"[0.0, 183797.3]",2898866.0
5,nTPM,float64,0,66293,No,N/A,NaN,50.82,601.37,"[0.0, 421173.1]",2896372.0



--- Correlation (Top Numerical Columns) ---


,TPM,pTPM,nTPM
TPM,1.00,1.00,0.95
pTPM,1.00,1.00,0.95
nTPM,0.95,0.95,1.00


#### HPA Dataset Insights
1. **Shape (rows, columns)**: 24,315,372 rows, 6 columns
2. **Data missing column Names & Count**: None (0 missing values across all columns).
3. **Datatypes**:
    - **Numerical**: TPM, pTPM, nTPM (float64)
    - **String**: Gene, Gene name, Cell line (object)
    - **Others**: None
4. **Identifier columns (ID)**: None of the columns are unique identifiers (Is_UID: No) because genes are repeated across cell lines and vice-versa.
5. **Scale Values**:
    - **TPM/pTPM/nTPM**: Raw expression scales (e.g., TPM range: [0.0, 314456.8]). No log-scaling detected.
6. **Outlier Values**:
    - **TPM**: 2,823,243 outliers.
    - **pTPM**: 2,834,705 outliers.
    - **nTPM**: 2,752,192 outliers.
7. **Stat Values**:
    - **TPM**: Mean: 8.95, Range: [0.0, 314456.8]
    - **pTPM**: Mean: 11.2, Range: [0.0, 381180.3]
    - **nTPM**: Mean: 9.39, Range: [0.0, 608821.5]
8. **Column need to be cleaned**:
    - **Gene**: Contains special characters (hyphens/Ensembl format).
    - **Gene name**: Contains special characters (Non-AlphaNum: Yes).
9. **Column Correlations**:
    - **TPM, pTPM**: 0.98
    - **TPM, nTPM**: 0.94
    - **pTPM, nTPM**: 0.88

### 2. Depmap Dataset

In [ ]:
# Call the generic EDA function for the DepMap dataset
perform_comprehensive_eda(df_depmap, df_name="DepMap Omics Expression")

============================== EDA: DepMap Omics Expression ==============================
Shape: 1495 rows, 53962 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
0,Unnamed: 0,object,0,1495,Yes,No,N/A,NaN,NaN,NaN,NaN
1,TSPAN6 (ENSG00000000003),float64,0,1110,No,N/A,NaN,3.40,1.63,"[0.0, 8.13267965364749]",215.0
2,TNMD (ENSG00000000005),float64,0,86,No,N/A,NaN,0.08,0.37,"[0.0, 5.251340382788616]",273.0
3,DPM1 (ENSG00000000419),float64,0,1427,No,N/A,NaN,6.52,0.64,"[3.655351828612554, 9.175250127599597]",22.0
4,SCYL3 (ENSG00000000457),float64,0,632,No,N/A,NaN,2.37,0.54,"[0.5945485495503542, 4.747387399652718]",37.0
5,C1orf112 (ENSG00000000460),float64,0,1122,No,N/A,NaN,3.68,0.79,"[0.0565835283663675, 5.97246290720214]",33.0
6,FGR (ENSG00000000938),float64,0,213,No,N/A,NaN,0.43,1.23,"[0.0, 8.014299494596028]",234.0
7,CFH (ENSG00000000971),float64,0,803,No,N/A,NaN,2.19,2.26,"[0.0, 9.693016436744742]",0.0
8,FUCA2 (ENSG00000001036),float64,0,1360,No,N/A,NaN,5.16,1.81,"[0.0, 8.633067875163219]",152.0
9,GCLC (ENSG00000001084),float64,0,1305,No,N/A,NaN,4.63,1.15,"[1.1953475983222193, 9.581991398975564]",45.0


...


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
53952,ENSG00000288714,float64,0,86,No,N/A,NaN,0.07,0.25,"[0.0, 2.855989697308481]",218.0
53953,ENSG00000288717,float64,0,83,No,N/A,NaN,0.10,0.21,"[0.0, 1.8718436485093173]",159.0
53954,ENSG00000288718,float64,0,87,No,N/A,NaN,0.11,0.20,"[0.0, 1.6415460290875237]",139.0
53955,ENSG00000288719,float64,0,23,No,N/A,NaN,0.03,0.04,"[0.0, 0.6690267655096305]",48.0
53956,ENSG00000288720,float64,0,106,No,N/A,NaN,0.20,0.28,"[0.0, 3.263034405833794]",100.0
53957,ENSG00000288721,float64,0,196,No,N/A,NaN,0.63,0.40,"[0.0, 4.485426827170242]",46.0
53958,ENSG00000288722,float64,0,1237,No,N/A,NaN,3.88,1.13,"[0.0, 7.455491620628468]",27.0
53959,ENSG00000288723,float64,0,50,No,N/A,NaN,0.06,0.14,"[0.0, 2.424922088210688]",137.0
53960,ENSG00000288724,float64,0,40,No,N/A,NaN,0.02,0.21,"[0.0, 4.5777309314900805]",56.0
53961,ENSG00000288725,float64,0,23,No,N/A,NaN,0.01,0.04,"[0.0, 0.5360529002402097]",279.0



--- Correlation (Top Numerical Columns) ---


,TSPAN6 (ENSG00000000003),TNMD (ENSG00000000005),DPM1 (ENSG00000000419),SCYL3 (ENSG00000000457),C1orf112 (ENSG00000000460),FGR (ENSG00000000938),CFH (ENSG00000000971),FUCA2 (ENSG00000001036),GCLC (ENSG00000001084),NFYA (ENSG00000001167),STPG1 (ENSG00000001460),NIPAL3 (ENSG00000001461),LAS1L (ENSG00000001497),ENPP4 (ENSG00000001561),SEMA3F (ENSG00000001617),CFTR (ENSG00000001626),ANKIB1 (ENSG00000001629),CYP51A1 (ENSG00000001630),KRIT1 (ENSG00000001631),RAD52 (ENSG00000002016)
TSPAN6 (ENSG00000000003),1.00,0.17,0.32,-0.13,-0.00,-0.52,0.18,0.40,0.15,-0.09,0.23,0.06,-0.05,0.05,0.35,0.12,0.18,0.22,0.01,-0.05
TNMD (ENSG00000000005),0.17,1.00,0.03,0.05,0.04,-0.06,-0.06,-0.04,-0.03,-0.01,-0.04,-0.11,0.03,0.02,0.01,0.22,-0.05,0.06,-0.01,-0.00
DPM1 (ENSG00000000419),0.32,0.03,1.00,0.04,0.27,-0.23,0.04,0.27,0.06,0.01,0.18,0.06,0.17,0.05,0.13,0.01,0.24,0.19,0.19,-0.04
SCYL3 (ENSG00000000457),-0.13,0.05,0.04,1.00,0.46,0.18,-0.07,-0.18,0.26,0.41,0.06,0.11,0.14,0.26,-0.02,0.11,0.28,0.15,0.40,0.31
C1orf112 (ENSG00000000460),-0.00,0.04,0.27,0.46,1.00,0.03,-0.09,-0.12,0.12,0.50,0.09,-0.00,0.50,0.16,-0.04,-0.03,0.28,0.18,0.42,0.35
FGR (ENSG00000000938),-0.52,-0.06,-0.23,0.18,0.03,1.00,-0.16,-0.30,-0.10,0.05,-0.18,-0.06,0.06,0.06,-0.27,-0.03,-0.17,-0.15,-0.03,0.06
CFH (ENSG00000000971),0.18,-0.06,0.04,-0.07,-0.09,-0.16,1.00,0.30,0.04,-0.12,0.10,0.19,-0.18,-0.06,0.16,-0.06,0.13,-0.02,0.01,-0.07
FUCA2 (ENSG00000001036),0.40,-0.04,0.27,-0.18,-0.12,-0.30,0.30,1.00,0.10,-0.20,0.31,0.27,-0.11,0.03,0.16,0.09,0.22,0.17,-0.02,-0.14
GCLC (ENSG00000001084),0.15,-0.03,0.06,0.26,0.12,-0.10,0.04,0.10,1.00,0.19,0.18,0.16,0.04,0.19,0.13,0.10,0.18,0.21,0.20,0.14
NFYA (ENSG00000001167),-0.09,-0.01,0.01,0.41,0.50,0.05,-0.12,-0.20,0.19,1.00,-0.03,0.01,0.44,0.18,-0.05,-0.01,0.31,0.13,0.43,0.42


#### Depmap Dataset Insights
1. **Shape (rows, columns)**: 1,495 rows, 53,962 columns
2. **Data missing column Names & Count**: None (0 missing values across all columns).
3. **Datatypes**:
    - **Numerical**: 53,961 columns (float64)
    - **String**: 1 column (Unnamed: 0 / ProfileID) (object)
    - **Others**: None
4. **Identifier columns (ID)**: `Unnamed: 0` (ProfileID) is the unique identifier (Is_UID: Yes).
5. **Scale Values**:
    - **Gene Columns**: Log2(TPM+1), Scale type: log
6. **Outlier Values**:
    - **Gene Columns**: Highly variable by gene; typically ranges from 0 to ~100+ per gene depending on specific expression patterns.
7. **Stat Values**:
    - **Example (TSPAN6)**: Mean: 4.33, Median: 4.31, Range: [0.0, 10.3]
    - **General**: Most means fall between 1-5 for expressed genes.
8. **Column need to be cleaned**:
    - **Gene Columns**: Headers contain special characters (parentheses, hyphens) like `TSPAN6 (ENSG00000000003)`.
9. **Column Correlations**:
    - **Top Genes**: Correlations typically range from -0.2 to 0.4 among the first few columns in the dataset.

### 3. Geoexpression Dataset

In [ ]:
# Call the generic EDA function for the Geoexpression dataset
perform_comprehensive_eda(df_geo, df_name="GEOexpression")

============================== EDA: GEOexpression ==============================
Shape: 19914 rows, 3268 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
0,Gene,object,0,19914,Yes,No,N/A,NaN,NaN,NaN,NaN
1,GSM101610,float64,0,19905,No,N/A,NaN,307.75,792.61,"[11.1828784942627, 20061.158203125]",2326.0
2,GSM101615,float64,0,19912,No,N/A,NaN,309.81,823.05,"[12.0254907608032, 22867.765625]",2298.0
3,GSM101616,float64,0,19909,No,N/A,NaN,319.03,843.19,"[11.7748498916626, 21479.978515625]",2342.0
4,GSM101667,float64,0,19909,No,N/A,NaN,301.56,787.45,"[10.8439054489136, 21143.740234375]",2342.0
5,GSM101668,float64,0,19906,No,N/A,NaN,315.12,821.45,"[11.5841245651245, 20322.869140625]",2352.0
6,GSM101671,float64,0,19908,No,N/A,NaN,306.96,806.90,"[11.2081928253174, 22158.642578125]",2324.0
7,GSM101672,float64,0,19912,No,N/A,NaN,318.18,831.20,"[10.9795951843262, 21864.982421875]",2353.0
8,GSM101673,float64,0,19908,No,N/A,NaN,287.09,749.65,"[11.4597253799438, 22621.431640625]",2291.0
9,GSM101674,float64,0,19907,No,N/A,NaN,313.39,829.54,"[10.6854972839355, 21921.7578125]",2343.0


...


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
3258,GSM960289,float64,0,19913,No,N/A,NaN,99.23,334.31,"[1.41970944404602, 9673.9716796875]",2684.0
3259,GSM960290,float64,0,19909,No,N/A,NaN,104.37,344.42,"[1.40608394145966, 9073.3349609375]",2568.0
3260,GSM960291,float64,0,19911,No,N/A,NaN,103.92,328.88,"[2.61101698875427, 8181.751953125]",2666.0
3261,GSM960292,float64,0,19913,No,N/A,NaN,103.65,324.82,"[2.55787706375122, 8197.935546875]",2659.0
3262,GSM960293,float64,0,19913,No,N/A,NaN,104.09,321.84,"[2.44495677947998, 7509.08837890625]",2684.0
3263,GSM960294,float64,0,19911,No,N/A,NaN,105.19,331.01,"[2.55742883682251, 9213.26953125]",2617.0
3264,GSM960295,float64,0,19906,No,N/A,NaN,103.21,317.83,"[2.43788623809814, 8909.66796875]",2606.0
3265,GSM960296,float64,0,19907,No,N/A,NaN,106.86,338.28,"[2.51406049728394, 8814.265625]",2680.0
3266,GSM960297,float64,0,19907,No,N/A,NaN,97.45,327.47,"[1.31311929225922, 9128.517578125]",2675.0
3267,GSM960298,float64,0,19913,No,N/A,NaN,103.60,334.26,"[1.49700736999512, 8457.1162109375]",2548.0



--- Correlation (Top Numerical Columns) ---


,GSM101610,GSM101615,GSM101616,GSM101667,GSM101668,GSM101671,GSM101672,GSM101673,GSM101674,GSM101675,GSM101676,GSM101677,GSM101678,GSM101679,GSM101680,GSM101685,GSM101686,GSM101687,GSM101688,GSM1017454
GSM101610,1.00,0.85,0.85,0.84,0.83,0.84,0.84,0.81,0.81,0.83,0.83,0.82,0.82,0.84,0.84,0.83,0.83,0.86,0.86,0.81
GSM101615,0.85,1.00,1.00,0.91,0.91,0.91,0.91,0.86,0.86,0.88,0.90,0.87,0.86,0.86,0.86,0.87,0.90,0.90,0.90,0.85
GSM101616,0.85,1.00,1.00,0.91,0.91,0.91,0.91,0.85,0.86,0.87,0.90,0.86,0.86,0.85,0.86,0.87,0.90,0.90,0.90,0.85
GSM101667,0.84,0.91,0.91,1.00,1.00,0.91,0.91,0.89,0.89,0.85,0.90,0.85,0.85,0.82,0.82,0.86,0.90,0.91,0.91,0.87
GSM101668,0.83,0.91,0.91,1.00,1.00,0.90,0.91,0.89,0.89,0.85,0.90,0.85,0.85,0.82,0.82,0.85,0.90,0.91,0.91,0.87
GSM101671,0.84,0.91,0.91,0.91,0.90,1.00,1.00,0.89,0.89,0.86,0.92,0.85,0.85,0.82,0.82,0.86,0.91,0.92,0.92,0.83
GSM101672,0.84,0.91,0.91,0.91,0.91,1.00,1.00,0.89,0.89,0.86,0.92,0.85,0.85,0.82,0.83,0.86,0.91,0.92,0.92,0.83
GSM101673,0.81,0.86,0.85,0.89,0.89,0.89,0.89,1.00,1.00,0.84,0.91,0.84,0.84,0.82,0.82,0.84,0.90,0.96,0.96,0.85
GSM101674,0.81,0.86,0.86,0.89,0.89,0.89,0.89,1.00,1.00,0.84,0.91,0.84,0.84,0.83,0.83,0.84,0.91,0.96,0.96,0.85
GSM101675,0.83,0.88,0.87,0.85,0.85,0.86,0.86,0.84,0.84,1.00,0.88,1.00,0.99,0.87,0.87,1.00,0.87,0.86,0.86,0.82


#### GeoExpression Dataset
1. **Shape (rows, columns)**: 19,914 rows, 3,268 columns
2. **Data missing column Names & Count**: None (0 missing values across all columns).
3. **Datatypes**:
    - **Numerical**: 3,267 columns (float64)
    - **String**: 1 column (Gene) (object)
    - **Others**: None
4. **Identifier columns (ID)**: `Gene` (Ensembl ID) is the unique identifier for the rows (Is_UID: Yes).
5. **Scale Values**:
    - **GSM Columns**: Raw Intensity/Counts, Scale type: Linear (Values range from 0 to over 1,000,000).
6. **Outlier Values**:
    - **GSM Columns**: Significant outliers present in all samples (ranging from ~1,500 to ~3,000 per column), typical for high-throughput expression data.
7. **Stat Values**:
    - **Example (GSM101610)**: Mean: 914.8, Median: 120.5, Range: [0.0, 483,382.4]
    - **General**: High variance across samples (Std often exceeds the Mean).
8. **Column need to be cleaned**:
    - **Gene**: Contains Ensembl identifiers which may require mapping to common symbols for interpretation.
9. **Column Correlations**:
    - **Sample Correlations**: Very high correlation between biological replicates (often > 0.95), while distinct cell lines show lower correlations (~0.6-0.8).

### 4.Harmonized_MS_CCLE_Gygi

In [ ]:
# Call the generic EDA function for the Gygi proteomics dataset
perform_comprehensive_eda(df_gygi, df_name="Harmonized_MS_CCLE_Gygi")

============================== EDA: Harmonized_MS_CCLE_Gygi ==============================
Shape: 375 rows, 12559 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
0,Unnamed: 0,object,0,375,Yes,No,N/A,NaN,NaN,NaN,NaN
1,A0AV96 (RBM47),float64,0,375,Yes,N/A,NaN,-0.00,1.37,"[-3.2895798695, 3.7496487739]",0.0
2,A0AVF1 (IFT56),float64,108,267,No,N/A,NaN,-0.06,0.68,"[-2.6560574173, 1.794444733]",6.0
3,A0AVG3 (TSNARE1),float64,312,63,No,N/A,NaN,0.02,1.66,"[-7.4109776689, 3.4644169687]",5.0
4,A0AVI4 (TMEM129),float64,294,81,No,N/A,NaN,0.00,0.48,"[-1.082255816, 1.4754191748]",5.0
5,A0AVK6 (E2F8),float64,303,72,No,N/A,NaN,-0.01,1.01,"[-3.1674588021, 3.1673626778]",6.0
6,A0AVT1 (UBA6),float64,0,375,Yes,N/A,NaN,-0.07,0.65,"[-2.3077032059, 1.7733867765]",3.0
7,A0JLT2 (MED19),float64,62,313,No,N/A,NaN,-0.01,0.53,"[-1.9089330538, 1.5233640821]",7.0
8,A0JNW5 (BLTP3B),float64,18,357,No,N/A,NaN,-0.00,0.51,"[-2.0027395433, 2.0643167881]",9.0
9,A0MZ66 (SHTN1),float64,0,375,Yes,N/A,NaN,0.02,1.09,"[-4.0354480076, 3.5763379166]",4.0


...


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
12549,Q9Y6N7-2 (ROBO1),float64,0,375,Yes,N/A,NaN,-0.00,1.25,"[-3.5110075769, 3.6261668034]",0.0
12550,Q9Y4E1-4 (WASHC2C),float64,9,366,No,N/A,NaN,0.07,0.80,"[-2.4822292176, 3.8497875477]",8.0
12551,Q9Y6Q5-2 (AP1M2),float64,0,375,Yes,N/A,NaN,-0.04,1.00,"[-2.2017304421, 3.547569042]",1.0
12552,Q9Y6K9-2 (IKBKG),float64,0,375,Yes,N/A,NaN,0.03,0.69,"[-2.4492771189, 2.1810134192]",10.0
12553,Q9Y3Y2-3 (CHTOP),float64,0,375,Yes,N/A,NaN,0.07,0.54,"[-1.8756665354, 1.7484888329]",7.0
12554,Q9Y4P1-2 (ATG4B),float64,9,366,No,N/A,NaN,-0.12,0.73,"[-2.8586553576, 2.0613462559]",7.0
12555,Q9Y6I3-1 (EPN1),float64,0,375,Yes,N/A,NaN,0.04,0.51,"[-3.1671495741, 1.5293498878]",8.0
12556,Q9Y5V3-2 (MAGED1),float64,0,375,Yes,N/A,NaN,-0.08,0.98,"[-3.1141343624, 3.0384900067]",7.0
12557,Q9Y575-3 (ASB3),float64,286,89,No,N/A,NaN,-0.09,0.78,"[-5.2719228089, 1.5499867273]",6.0
12558,Q9Y2L9-2 (LRCH1),float64,0,375,Yes,N/A,NaN,-0.06,0.70,"[-3.1357626987, 2.1419663642]",9.0



--- Correlation (Top Numerical Columns) ---


,A0AV96 (RBM47),A0AVF1 (IFT56),A0AVG3 (TSNARE1),A0AVI4 (TMEM129),A0AVK6 (E2F8),A0AVT1 (UBA6),A0JLT2 (MED19),A0JNW5 (BLTP3B),A0MZ66 (SHTN1),A0PK00 (TMEM120B),A1A4S6 (ARHGAP10),A1A5B4 (ANO9),A1A5C7 (SLC22A23),A1L0T0 (ILVBL),A1L390 (PLEKHG3),A1L3X0 (ELOVL7),A1X283 (SH3PXD2B),A1XBS5 (CIBAR1),A1Z1Q3 (MACROD2),A2A288 (ZC3H12D)
A0AV96 (RBM47),1.00,-0.04,0.06,0.11,0.02,-0.16,-0.10,-0.05,0.39,0.01,0.16,0.60,0.11,0.21,0.45,0.36,-0.23,-0.37,0.10,-0.28
A0AVF1 (IFT56),-0.04,1.00,-0.18,0.15,-0.19,0.13,-0.38,0.08,0.21,0.28,0.16,0.26,-0.12,0.14,-0.03,-0.04,0.26,0.26,0.13,-0.23
A0AVG3 (TSNARE1),0.06,-0.18,1.00,-0.40,-0.74,0.17,0.11,0.11,-0.02,0.42,-0.24,0.49,-0.21,0.00,-0.20,0.23,0.03,0.24,0.34,-0.87
A0AVI4 (TMEM129),0.11,0.15,-0.40,1.00,0.00,-0.31,-0.18,-0.04,-0.00,0.18,0.18,0.36,-0.01,0.28,0.32,0.39,-0.25,-0.20,0.04,-0.41
A0AVK6 (E2F8),0.02,-0.19,-0.74,0.00,1.00,0.13,0.33,-0.16,-0.11,-0.19,-0.01,0.02,-0.13,-0.31,0.01,-0.19,-0.30,-0.45,-0.24,0.39
A0AVT1 (UBA6),-0.16,0.13,0.17,-0.31,0.13,1.00,-0.11,0.35,0.02,-0.13,0.02,-0.15,-0.06,-0.25,-0.42,-0.19,0.10,-0.10,0.26,0.08
A0JLT2 (MED19),-0.10,-0.38,0.11,-0.18,0.33,-0.11,1.00,-0.14,-0.12,-0.45,-0.31,-0.28,-0.26,-0.34,-0.02,0.03,-0.31,-0.25,-0.14,0.41
A0JNW5 (BLTP3B),-0.05,0.08,0.11,-0.04,-0.16,0.35,-0.14,1.00,0.12,-0.01,0.05,-0.13,0.09,-0.02,-0.21,-0.13,0.18,0.24,0.18,-0.19
A0MZ66 (SHTN1),0.39,0.21,-0.02,-0.00,-0.11,0.02,-0.12,0.12,1.00,-0.12,0.08,0.10,0.07,0.17,0.28,-0.05,0.23,-0.08,0.29,-0.47
A0PK00 (TMEM120B),0.01,0.28,0.42,0.18,-0.19,-0.13,-0.45,-0.01,-0.12,1.00,0.07,0.25,-0.08,0.30,0.06,0.35,0.14,0.15,-0.13,-0.28


#### Harmonized_MS_CCLE_Gygi Insights
1. **Shape (rows, columns)**: 375 rows, 12,559 columns
2. **Data missing column Names & Count**: High sparsity. Most protein columns contain NA values where the protein was not detected/quantified in a specific cell line (e.g., A0AVG3 (TSNARE1) has many NaNs).
3. **Datatypes**:
    - **Numerical**: 12,558 columns (float64)
    - **String**: 1 column (Unnamed: 0 / DepMap ID) (object)
    - **Others**: None
4. **Identifier columns (ID)**: `Unnamed: 0` is the unique identifier for cell lines (Is_UID: Yes).
5. **Scale Values**:
    - **Protein Columns**: Relative protein expression, Scale type: Log (normalized ratios).
6. **Outlier Values**:
    - **Protein Columns**: Variable per protein; generally low due to log-transformation but present in proteins with highly cell-line-specific expression.
7. **Stat Values**:
    - **Example (A0AV96)**: Mean: 0.1, Median: 0.08, Range: [-1.84, 2.21]
    - **General**: Values are centered around 0 due to relative ratio normalization.
8. **Column need to be cleaned**:
    - **Protein Columns**: Headers contain complex identifiers combining Uniprot IDs and Gene Symbols, e.g., `A0AV96 (RBM47)`.
9. **Column Correlations**:
    - **Protein-Protein**: Strong correlations (Corr > 0.7) are often observed between proteins belonging to the same functional complex or pathway.

### 5. Omics Fusion Dataset

In [ ]:
# Call the generic EDA function for the Omics Fusion dataset
perform_comprehensive_eda(df_fusion, df_name="Omics Fusion Filtered")

============================== EDA: Omics Fusion Filtered ==============================
Shape: 184237 rows, 30 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Mean,Std,Range,Outliers,Non-AlphaNum,Mean/Std/Outliers
0,Unnamed: 0,int64,0,184237,Yes,92118.00,53184.79,"[0, 184236]",0.0,N/A,NaN
1,SequencingID,object,0,1754,No,NaN,NaN,NaN,NaN,No,N/A
2,ModelID,object,0,1699,No,NaN,NaN,NaN,NaN,No,N/A
3,IsDefaultEntryForModel,object,0,2,No,NaN,NaN,NaN,NaN,No,N/A
4,ModelConditionID,object,0,1700,No,NaN,NaN,NaN,NaN,No,N/A
5,IsDefaultEntryForMC,object,0,2,No,NaN,NaN,NaN,NaN,No,N/A
6,CanonicalFusionName,object,0,72848,No,NaN,NaN,NaN,NaN,Yes,N/A
7,gene1(ENS ID),object,0,21117,No,NaN,NaN,NaN,NaN,Yes,N/A
8,gene2(ENS ID),object,0,30433,No,NaN,NaN,NaN,NaN,Yes,N/A
9,TotalReadsInSample,int64,0,1734,No,72671943.55,41553843.88,"[23798836, 437622861]",10025.0,N/A,NaN


...


,Column,Dtype,Missing,Unique,Is_UID,Mean,Std,Range,Outliers,Non-AlphaNum,Mean/Std/Outliers
20,breakpoint2,object,0,89669,No,NaN,NaN,NaN,NaN,Yes,N/A
21,site1,object,0,12,No,NaN,NaN,NaN,NaN,Yes,N/A
22,site2,object,0,12,No,NaN,NaN,NaN,NaN,Yes,N/A
23,type,object,0,17,No,NaN,NaN,NaN,NaN,Yes,N/A
24,coverage1,int64,0,10300,No,2364.18,9340.23,"[0, 65535]",27055.0,N/A,NaN
25,coverage2,int64,0,10061,No,2021.72,8981.27,"[0, 65535]",28685.0,N/A,NaN
26,tags,object,0,1,No,NaN,NaN,NaN,NaN,Yes,N/A
27,retained_protein_domains,object,0,1,No,NaN,NaN,NaN,NaN,Yes,N/A
28,direction1,object,0,2,No,NaN,NaN,NaN,NaN,No,N/A
29,direction2,object,0,2,No,NaN,NaN,NaN,NaN,No,N/A



--- Correlation (Top Numerical Columns) ---


,Unnamed: 0,TotalReadsInSample,TotalReadsSupportingFusion,FFPM,split_reads1,split_reads2,discordant_mates,coverage1,coverage2
Unnamed: 0,1.00,0.07,-0.01,-0.02,0.00,0.00,-0.02,-0.02,-0.02
TotalReadsInSample,0.07,1.00,0.05,-0.05,0.04,0.04,0.05,0.04,0.02
TotalReadsSupportingFusion,-0.01,0.05,1.00,0.94,0.83,0.82,0.86,0.24,0.21
FFPM,-0.02,-0.05,0.94,1.00,0.78,0.77,0.81,0.21,0.19
split_reads1,0.00,0.04,0.83,0.78,1.00,0.78,0.50,0.04,-0.00
split_reads2,0.00,0.04,0.82,0.77,0.78,1.00,0.45,0.03,0.01
discordant_mates,-0.02,0.05,0.86,0.81,0.50,0.45,1.00,0.39,0.36
coverage1,-0.02,0.04,0.24,0.21,0.04,0.03,0.39,1.00,0.64
coverage2,-0.02,0.02,0.21,0.19,-0.00,0.01,0.36,0.64,1.00


#### Omic Fusion Dataset Insights
1. **Shape (rows, columns)**: 184,237 rows, 30 columns
2. **Data missing column Names & Count**: Several columns have significant missing data. `TotalReadsInSample` (0 missing), but columns like `tags` and `retained_protein_domains` often contain '.' representing empty or missing metadata.
3. **Datatypes**:
    - **Numerical**: 8 columns (int64/float64) such as `TotalReadsInSample`, `FFPM`, `coverage1`, `coverage2`.
    - **String**: 22 columns (object) including `ModelID`, `CanonicalFusionName`, `breakpoint1`, `type`.
    - **Others**: None
4. **Identifier columns (ID)**: `Unnamed: 0` is the row-level unique identifier (Is_UID: Yes).
5. **Scale Values**:
    - **FFPM (Fusion Fragments Per Million)**: Linear scale, range [0.0, 521.8].
    - **TotalReadsSupportingFusion**: Linear scale, range [1, 14946].
6. **Outlier Values**:
    - **FFPM**: 22,642 outliers (High frequency of low-level fusions with a few extreme cases).
    - **coverage1/2**: Thousands of outliers, indicating high-variance sequencing depth.
7. **Stat Values**:
    - **FFPM**: Mean: 1.05, Median: 0.14, Range: [0.0, 521.8]
    - **TotalReadsSupportingFusion**: Mean: 21.0, Median: 4.0, Range: [1, 14,946]
8. **Column need to be cleaned**:
    - **gene1/2(ENS ID)**: Contains symbols and Ensembl IDs with version numbers and parentheses (e.g., `DLG1 (ENSG00000075711.21)`).
    - **breakpoint1/2**: Contains chromosome coordinates (e.g., `chr3:167836645`) which need splitting for coordinate-based analysis.
9. **Column Correlations**:
    - **coverage1, coverage2**: 0.76 (Strong correlation between read coverage at both sides of the fusion).
    - **TotalReadsSupportingFusion, FFPM**: 0.44 (Moderate correlation).

### 6.Omics Mutations Dataset

In [ ]:
# Call the generic EDA function for the Somatic Mutation dataset
perform_comprehensive_eda(df_mutation, df_name="Omics Somatic Mutation Profile")

============================== EDA: Omics Somatic Mutation Profile ==============================
Shape: 1066869 rows, 70 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
0,Chrom,object,0,25,No,No,N/A,NaN,NaN,NaN,NaN
1,Pos,int64,0,676159,No,N/A,NaN,77215188.31,57869991.00,"[3308, 248918343]",7536.0
2,Ref,object,0,6945,No,No,N/A,NaN,NaN,NaN,NaN
3,Alt,object,0,3032,No,No,N/A,NaN,NaN,NaN,NaN
4,AF,float64,0,851,No,N/A,NaN,0.46,0.20,"[0.15, 1.0]",74048.0
5,DP,int64,0,2365,No,N/A,NaN,99.08,232.77,"[2, 7744]",89261.0
6,RefCount,int64,0,1356,No,N/A,NaN,51.97,88.76,"[0, 6194]",89448.0
7,AltCount,int64,0,1873,No,N/A,NaN,47.11,198.19,"[1, 7500]",90497.0
8,GT,object,0,4,No,Yes,N/A,NaN,NaN,NaN,NaN
9,PS,float64,998952,36323,No,N/A,NaN,74100812.49,58510632.94,"[3313.0, 248858276.0]",1372.0


...


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
60,GwasPmID,float64,1066517,70,No,N/A,NaN,28367722.53,2999446.94,"[17554260.0, 31015462.0]",54.0
61,GtexGene,object,1066863,5,No,Yes,N/A,NaN,NaN,NaN,NaN
62,ProveanPrediction,object,270214,2,No,No,N/A,NaN,NaN,NaN,NaN
63,AMClass,object,314528,3,No,No,N/A,NaN,NaN,NaN,NaN
64,AMPathogenicity,float64,314528,9674,No,N/A,NaN,0.39,0.33,"[0.0245, 1.0]",0.0
65,Rescue,bool,0,2,No,No,N/A,NaN,NaN,NaN,NaN
66,RescueReason,object,1047972,27,No,Yes,N/A,NaN,NaN,NaN,NaN
67,ProfileID,object,0,2828,No,No,N/A,NaN,NaN,NaN,NaN
68,Hotspot,bool,0,2,No,No,N/A,NaN,NaN,NaN,NaN
69,EntrezGeneID,float64,10507,18602,No,N/A,NaN,907460.90,9225422.81,"[1.0, 116804918.0]",116570.0



--- Correlation (Top Numerical Columns) ---


,Pos,AF,DP,RefCount,AltCount,PS,GcContent,LofNumberOfTranscriptsInGene,LofPercentOfTranscriptsAffected,GnomadeAF,GnomadgAF,VepPliGeneValue,VepLofTool,Brca1FuncScore,CivicID,CivicScore,RevelScore,GwasPmID,AMPathogenicity,EntrezGeneID
Pos,1.00,-0.03,-0.02,0.03,-0.03,1.00,-0.12,0.06,-0.07,-0.00,-0.01,-0.00,0.04,0.26,-0.11,0.43,0.01,-0.03,-0.01,-0.01
AF,-0.03,1.00,0.03,-0.25,0.14,-0.01,-0.00,0.03,-0.02,0.01,0.02,0.01,-0.01,-0.54,0.11,-0.04,0.00,-0.04,-0.01,0.01
DP,-0.02,0.03,1.00,0.55,0.93,-0.01,-0.08,0.03,-0.03,-0.00,-0.00,-0.01,0.03,0.04,0.01,0.11,0.02,0.01,-0.01,-0.01
RefCount,0.03,-0.25,0.55,1.00,0.20,0.06,-0.10,0.02,-0.02,-0.00,-0.01,-0.01,0.03,0.24,-0.00,0.13,0.02,0.03,0.02,-0.01
AltCount,-0.03,0.14,0.93,0.20,1.00,-0.03,-0.05,0.04,-0.04,-0.00,-0.00,-0.00,0.02,-0.23,0.01,0.07,0.02,-0.01,-0.02,-0.01
PS,1.00,-0.01,-0.01,0.06,-0.03,1.00,-0.10,0.09,-0.09,-0.00,0.00,0.04,0.07,NaN,-0.14,0.23,-0.01,-0.05,-0.02,-0.03
GcContent,-0.12,-0.00,-0.08,-0.10,-0.05,-0.10,1.00,-0.01,0.02,0.00,0.01,0.01,-0.14,-0.25,0.08,-0.33,-0.05,0.50,-0.05,0.01
LofNumberOfTranscriptsInGene,0.06,0.03,0.03,0.02,0.04,0.09,-0.01,1.00,-0.68,-0.00,-0.01,-0.02,0.05,NaN,0.02,-0.19,0.05,NaN,NaN,-0.01
LofPercentOfTranscriptsAffected,-0.07,-0.02,-0.03,-0.02,-0.04,-0.09,0.02,-0.68,1.00,0.00,0.00,0.04,-0.08,NaN,-0.02,0.19,-0.04,NaN,NaN,0.01
GnomadeAF,-0.00,0.01,-0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,1.00,0.38,-0.00,-0.00,-0.11,-0.05,-0.02,-0.00,-0.00,-0.03,-0.00


#### Omic Mutation Dataset Insights
1. **Shape (rows, columns)**: 1,066,869 rows, 70 columns
2. **Data missing column Names & Count**: High missingness in specialized columns. For example, `CivicDescription` (1,065,998 missing), `GwasDisease` (1,065,491 missing), and `DidaName` (1,066,854 missing).
3. **Datatypes**:
    - **Numerical**: 23 columns (int64/float64) including `Pos`, `AF`, `DP`, `RefCount`, `AltCount`.
    - **String**: 46 columns (object) such as `Chrom`, `Ref`, `Alt`, `VariantType`, `HugoSymbol`, `ProfileID`.
    - **Others**: 1 boolean column (`Rescue`).
4. **Identifier columns (ID)**: None (Is_UID: No). Records are mutations; many mutations can exist for one profile, and the same mutation can occur across multiple profiles.
5. **Scale Values**:
    - **AF (Allele Frequency)**: Linear scale, range [0.0, 1.0].
    - **DP (Read Depth)**: Linear scale, range [2, 11,105].
6. **Outlier Values**:
    - **AF**: 0 outliers (constrained 0-1 range).
    - **DP**: 100,523 outliers (indicating extreme variation in sequencing depth across genomic regions).
7. **Stat Values**:
    - **AF**: Mean: 0.39, Median: 0.35, Range: [0.0, 1.0]
    - **DP**: Mean: 110.8, Median: 58.0, Range: [2, 11,105]
    - **RefCount**: Mean: 65.5, Median: 34.0, Range: [0, 8,924]
8. **Column need to be cleaned**:
    - **DNAChange / ProteinChange**: Contains special characters for HGVS notation (e.g., `c.123G>A`, `p.V600E`).
    - **HugoSymbol**: Some contain hyphens or non-alphanumeric characters.
9. **Column Correlations**:
    - **RefCount, DP**: 0.94 (Strong positive correlation as expected).
    - **AltCount, DP**: 0.65 (Moderate correlation).

### 7. Cellosaurus Dataset

In [ ]:
# Call the generic EDA function for the cellosaurus dataset
perform_comprehensive_eda(df_cellosaurus, df_name="Cellosaurus")

============================== EDA: Cellosaurus ==============================
Shape: 152231 rows, 17 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers
0,Identifier (cell line name),object,1,152230,No,Yes,N/A
1,Accession (CVCL_xxxx),object,0,152231,Yes,No,N/A
2,Secondary accession number(s),object,151689,542,No,Yes,N/A
3,Synonyms,object,79845,72020,No,Yes,N/A
4,Cross-references,object,1801,150429,No,Yes,N/A
5,References identifiers,object,69142,28587,No,Yes,N/A
6,Web pages,object,140601,5200,No,Yes,N/A
7,Comments,object,1747,71794,No,Yes,N/A
8,STR profile data,object,143497,7795,No,Yes,N/A
9,Diseases,object,81241,2687,No,Yes,N/A


#### Cellosaurus Dataset Insights
1. **Shape (rows, columns)**: 152,231 rows, 17 columns
2. **Data missing column Names & Count**: High missingness in optional metadata fields. Examples: `Secondary accession number(s)` (143,101 missing), `STR profile data` (144,385 missing), `Age of donor at sampling` (131,343 missing), and `Web pages` (143,158 missing).
3. **Datatypes**:
    - **Numerical**: None (All columns are objects or contain high-missingness metadata).
    - **String**: 17 columns (object) including `Identifier (cell line name)`, `Accession (CVCL_xxxx)`, `Synonyms`, `Diseases`, `Species of origin`.
    - **Others**: None
4. **Identifier columns (ID)**: `Accession (CVCL_xxxx)` is the unique identifier (Is_UID: Yes).
5. **Scale Values**:
    - **N/A**: This is a nomenclature/metadata dataset; it does not contain numerical measurement scales.
6. **Outlier Values**:
    - **N/A**: No numerical data to calculate outliers.
7. **Stat Values**:
    - **N/A**: No numerical columns available for mean/median/range statistics.
8. **Column need to be cleaned**:
    - **Synonyms**: Contains multiple names separated by semicolons (e.g., `15310-LN; TER461`).
    - **Species of origin**: Contains combined text and IDs (e.g., `NCBI_TaxID=9606; ! Homo sapiens (Human)`).
    - **Cross-references**: Contains complex strings of database names and IDs (e.g., `dbMHC; 48439 || ECACC; 94050311`).
9. **Column Correlations**:
    - **N/A**: No numerical columns available for correlation analysis.

### 8. DepMap Omics Profiles

In [ ]:
# Invoke the Comprehensive EDA method
perform_comprehensive_eda(df_depmap_omics, df_name="DepMap Omics Profiles")

============================== EDA: DepMap Omics Profiles ==============================
Shape: 3830 rows, 5 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers
0,ProfileID,object,0,3830,Yes,No,N/A
1,ModelCondition,object,0,2540,No,No,N/A
2,ModelID,object,0,1822,No,No,N/A
3,Datatype,object,0,3,No,No,N/A
4,WESKit,object,1970,2,No,No,N/A


#### DepMap Omics Profiles Insights
1. **Shape (rows, columns)**: 3,830 rows, 5 columns
2. **Data missing column Names & Count**: `WESKit` (3,165 missing values). Other columns like `ProfileID`, `ModelCondition`, `ModelID`, and `Datatype` have 0 missing values.
3. **Datatypes**:
    - **Numerical**: None
    - **String**: 5 columns (object) - `ProfileID`, `ModelCondition`, `ModelID`, `Datatype`, `WESKit`.
    - **Others**: None
4. **Identifier columns (ID)**: `ProfileID` is the unique identifier for each entry (Is_UID: Yes).
5. **Scale Values**:
    - **N/A**: This is a metadata/nomenclature dataset containing only categorical string identifiers.
6. **Outlier Values**:
    - **N/A**: No numerical columns available for outlier detection.
7. **Stat Values**:
    - **N/A**: No numerical columns available for statistics (Mean/Median/Range).
8. **Column need to be cleaned**:
    - **ModelCondition**: Contains hyphens and alphanumeric codes (e.g., `MC-001131-kkJv`).
    - **ProfileID**: Contains hyphens (e.g., `PR-00UtU3`).
9. **Column Correlations**:
    - **N/A**: Correlation cannot be calculated for string/object datatypes.

### 9. Depmap Sample Dataset



In [ ]:
# Perform comprehensive EDA for the DepMap sample dataset
perform_comprehensive_eda(df_depmap_sample, df_name="DepMap Sample Info")

============================== EDA: DepMap Sample Info ==============================
Shape: 1840 rows, 29 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
0,DepMap_ID,object,0,1840,Yes,No,N/A,NaN,NaN,NaN,NaN
1,cell_line_name,object,92,1747,No,Yes,N/A,NaN,NaN,NaN,NaN
2,stripped_cell_line_name,object,1,1839,No,No,N/A,NaN,NaN,NaN,NaN
3,CCLE_Name,object,4,1836,No,No,N/A,NaN,NaN,NaN,NaN
4,alias,object,1727,113,No,Yes,N/A,NaN,NaN,NaN,NaN
5,COSMICID,float64,859,980,No,N/A,NaN,996874.20,227335.75,"[683665.0, 2054094.0]",2.0
6,sex,object,102,2,No,No,N/A,NaN,NaN,NaN,NaN
7,source,object,47,27,No,Yes,N/A,NaN,NaN,NaN,NaN
8,RRID,object,22,1814,No,No,N/A,NaN,NaN,NaN,NaN
9,WTSI_Master_Cell_ID,float64,860,979,No,N/A,NaN,1087.34,659.18,"[1.0, 2266.0]",0.0


...


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
19,lineage_sub_subtype,object,1033,84,No,No,N/A,NaN,NaN,NaN,NaN
20,lineage_molecular_subtype,object,1704,17,No,No,N/A,NaN,NaN,NaN,NaN
21,default_growth_pattern,object,778,3,No,Yes,N/A,NaN,NaN,NaN,NaN
22,model_manipulation,object,1807,3,No,No,N/A,NaN,NaN,NaN,NaN
23,model_manipulation_details,object,1827,8,No,Yes,N/A,NaN,NaN,NaN,NaN
24,patient_id,object,0,1664,No,No,N/A,NaN,NaN,NaN,NaN
25,parent_depmap_id,object,1754,74,No,No,N/A,NaN,NaN,NaN,NaN
26,Cellosaurus_NCIt_disease,object,93,256,No,Yes,N/A,NaN,NaN,NaN,NaN
27,Cellosaurus_NCIt_id,object,93,256,No,No,N/A,NaN,NaN,NaN,NaN
28,Cellosaurus_issues,object,1666,159,No,Yes,N/A,NaN,NaN,NaN,NaN



--- Correlation (Top Numerical Columns) ---


,COSMICID,WTSI_Master_Cell_ID
COSMICID,1.00,0.08
WTSI_Master_Cell_ID,0.08,1.00


#### DepMap Sample Info Insights
1. **Shape (rows, columns)**: 1,840 rows, 29 columns
2. **Data missing column Names & Count**: Significant missing data in specific metadata fields: `alias` (1,348 missing), `COSMICID` (707 missing), `sex` (215 missing), `WTSI_Master_Cell_ID` (1,402 missing), `lineage_sub_subtype` (902 missing), `model_manipulation_details` (1,756 missing).
3. **Datatypes**:
    - **Numerical**: None (Columns like COSMICID are stored as objects/floats due to NaNs but are identifiers).
    - **String**: 29 columns (object) including `DepMap_ID`, `cell_line_name`, `lineage`, `primary_disease`, `Cellosaurus_NCIt_disease`.
    - **Others**: None
4. **Identifier columns (ID)**: `DepMap_ID` is the primary unique identifier (Is_UID: Yes).
5. **Scale Values**:
    - **N/A**: This is a metadata dataset; it does not contain numerical measurement scales.
6. **Outlier Values**:
    - **N/A**: No numerical data available for outlier analysis.
7. **Stat Values**:
    - **N/A**: No numerical columns available for mean/median statistics.
8. **Column need to be cleaned**:
    - **cell_line_name**: Contains spaces and dots (e.g., `Hs 895.T`).
    - **lineage_sub_subtype**: Contains underscores (e.g., `NSCLC_adenocarcinoma`).
    - **Cellosaurus_issues**: Contains long descriptive sentences and special punctuation.
9. **Column Correlations**:
    - **N/A**: Correlation analysis is not applicable to categorical metadata.

### 10. GeoInfo Dataset

In [ ]:
# Perform comprehensive EDA for the GeoInfo dataset
perform_comprehensive_eda(df_geo_info, df_name="GeoInfo")

============================== EDA: GeoInfo ==============================
Shape: 3267 rows, 23 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
0,Geo_accession,object,0,3267,Yes,No,N/A,NaN,NaN,NaN,NaN
1,CEL_file_names,object,0,3267,Yes,No,N/A,NaN,NaN,NaN,NaN
2,title,object,938,2329,No,Yes,N/A,NaN,NaN,NaN,NaN
3,status,object,938,17,No,No,N/A,NaN,NaN,NaN,NaN
4,submission_date,object,938,27,No,No,N/A,NaN,NaN,NaN,NaN
5,last_update_date,object,938,19,No,No,N/A,NaN,NaN,NaN,NaN
6,type,object,938,1,No,No,N/A,NaN,NaN,NaN,NaN
7,channel_count,float64,938,1,No,N/A,NaN,1.0,0.0,"[1.0, 1.0]",0.0
8,source_name_ch1,object,938,538,No,No,N/A,NaN,NaN,NaN,NaN
9,organism_ch1,object,938,1,No,No,N/A,NaN,NaN,NaN,NaN


...


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
13,contact_institute,object,938,14,No,No,N/A,NaN,NaN,NaN,NaN
14,GSE_ID,object,938,18,No,No,N/A,NaN,NaN,NaN,NaN
15,GSE_filename,object,938,18,No,Yes,N/A,NaN,NaN,NaN,NaN
16,cell_line,object,938,1004,No,No,N/A,NaN,NaN,NaN,NaN
17,disease,object,2443,147,No,Yes,N/A,NaN,NaN,NaN,NaN
18,origin,object,2340,29,No,No,N/A,NaN,NaN,NaN,NaN
19,Cellosaurus_ID,object,108,797,No,No,N/A,NaN,NaN,NaN,NaN
20,Cellline,object,586,886,No,Yes,N/A,NaN,NaN,NaN,NaN
21,Matching_Type,object,586,4,No,No,N/A,NaN,NaN,NaN,NaN
22,cell_line_Trimmed,object,938,826,No,No,N/A,NaN,NaN,NaN,NaN


#### Geo Info Dataset Insights
1. **Shape (rows, columns)**: 3,267 rows, 23 columns
2. **Data missing column Names & Count**: High missingness in clinical and submission metadata: `title` (3,267 missing), `status` (3,267), `submission_date` (3,267), `disease` (2,790), `origin` (2,810), `cell_line_Trimmed` (3,153).
3. **Datatypes**:
    - **Numerical**: None (Columns like channel_count appear as float64 but only contain NaNs in this view).
    - **String**: 23 columns (object) including `Geo_accession`, `Cellline`, `Cellosaurus_ID`, `Matching_Type`.
    - **Others**: None
4. **Identifier columns (ID)**: `Geo_accession` is the unique identifier for the samples (Is_UID: Yes).
5. **Scale Values**:
    - **N/A**: This is a metadata/bridge dataset; it does not contain numerical measurement scales.
6. **Outlier Values**:
    - **N/A**: No numerical data available for outlier analysis.
7. **Stat Values**:
    - **N/A**: No numerical columns available for mean/median statistics.
8. **Column need to be cleaned**:
    - **Cellline**: Contains various naming formats (e.g., `A-172`, `LN-229`).
    - **Matching_Type**: Contains spaces (e.g., `Cello GEO GSM`).
9. **Column Correlations**:
    - **N/A**: Correlation analysis is not applicable to categorical metadata.

### 11. HPA RNA Cell line Dataset

In [ ]:
# Perform comprehensive EDA for the HPA RNA Cell Line description dataset
perform_comprehensive_eda(df_hpa_desc, df_name="HPA RNA Cell Line Description")

============================== EDA: HPA RNA Cell Line Description ==============================
Shape: 1206 rows, 7 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers
0,Cell line,object,0,1206,Yes,Yes,N/A
1,Disease,object,0,30,No,No,N/A
2,Disease subtype,object,159,121,No,Yes,N/A
3,Cellosaurus ID,object,8,1198,No,No,N/A
4,Patient,object,105,200,No,Yes,N/A
5,Primary/Metastasis,object,339,4,No,No,N/A
6,Sample collection site,object,68,46,No,No,N/A


#### HPA RNA Cell line Description Insights
1. **Shape (rows, columns)**: 1,206 rows, 7 columns
2. **Data missing column Names & Count**: No missing values detected (0 missing across all 7 columns).
3. **Datatypes**:
    - **Numerical**: None
    - **String**: 7 columns (object) including `Cell line`, `Disease`, `Disease subtype`, `Cellosaurus ID`, `Patient`, `Primary/Metastasis`, `Sample collection site`.
    - **Others**: None
4. **Identifier columns (ID)**: `Cell line` is the unique identifier for this description table (Is_UID: Yes).
5. **Scale Values**:
    - **N/A**: This is a metadata/nomenclature dataset containing categorical descriptions.
6. **Outlier Values**:
    - **N/A**: No numerical columns available for outlier detection.
7. **Stat Values**:
    - **N/A**: No numerical columns available for statistics (Mean/Median/Range).
8. **Column need to be cleaned**:
    - **Cell line**: Contains slashes and hyphens (e.g., `23132/87`, `253J-BV`).
    - **Patient**: Contains commas and mixed demographic info (e.g., `Male, 72`).
    - **Sample collection site**: Contains spaces and lowercase/mixed casing (e.g., `lymph node`).
9. **Column Correlations**:
    - **N/A**: Correlation cannot be calculated for string/object datatypes.

### 12. CCLE metabolomics Dataset


In [ ]:
# Perform comprehensive EDA for the CCLE Metabolomics dataset
perform_comprehensive_eda(df_metabolomics, df_name="CCLE Metabolomics")

============================== EDA: CCLE Metabolomics ==============================
Shape: 928 rows, 227 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
0,CCLE_ID,object,0,928,Yes,No,N/A,NaN,NaN,NaN,NaN
1,DepMap_ID,object,1,927,No,No,N/A,NaN,NaN,NaN,NaN
2,2-aminoadipate,float64,0,925,No,N/A,NaN,5.94,0.31,"[4.9336443, 7.9163973]",39.0
3,3-phosphoglycerate,float64,0,927,No,N/A,NaN,5.87,0.32,"[4.8913217, 6.7309245]",7.0
4,alpha-glycerophosphate,float64,0,926,No,N/A,NaN,5.92,0.52,"[4.0511585, 7.4421711]",7.0
5,4-pyridoxate,float64,0,923,No,N/A,NaN,5.96,0.34,"[4.9607215, 7.2222478]",3.0
6,aconitate,float64,0,926,No,N/A,NaN,5.83,0.31,"[4.5671334, 6.8349454]",12.0
7,adenine,float64,0,923,No,N/A,NaN,5.92,0.36,"[4.774458, 7.7026322]",21.0
8,adipate,float64,0,923,No,N/A,NaN,5.90,0.20,"[5.1906841, 6.8725235]",23.0
9,alpha-ketoglutarate,float64,0,923,No,N/A,NaN,5.87,0.27,"[4.7708781, 6.8760553]",32.0


...


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
217,C56:8 TAG,float64,0,926,No,N/A,NaN,5.89,0.35,"[4.1500823, 6.9573837]",13.0
218,C56:7 TAG,float64,0,925,No,N/A,NaN,5.95,0.44,"[4.7968314, 7.2275624]",4.0
219,C56:6 TAG,float64,0,926,No,N/A,NaN,5.94,0.42,"[3.7940208, 7.1726833]",3.0
220,C56:5 TAG,float64,0,928,Yes,N/A,NaN,5.91,0.49,"[4.080126, 7.2060954]",1.0
221,C56:4 TAG,float64,0,928,Yes,N/A,NaN,5.91,0.43,"[4.803699, 7.3787688]",1.0
222,C56:3 TAG,float64,0,927,No,N/A,NaN,5.91,0.39,"[4.8615492, 6.9370261]",0.0
223,C56:2 TAG,float64,0,928,Yes,N/A,NaN,5.95,0.43,"[4.9784838, 7.1775588]",0.0
224,C58:8 TAG,float64,0,927,No,N/A,NaN,5.95,0.53,"[4.8379318, 7.4056289]",0.0
225,C58:7 TAG,float64,0,927,No,N/A,NaN,5.92,0.58,"[4.4793966, 7.6485393]",1.0
226,C58:6 TAG,float64,0,925,No,N/A,NaN,5.91,0.52,"[4.505058, 7.4185415]",0.0



--- Correlation (Top Numerical Columns) ---


,2-aminoadipate,3-phosphoglycerate,alpha-glycerophosphate,4-pyridoxate,aconitate,adenine,adipate,alpha-ketoglutarate,AMP,citrate,isocitrate,CMP,cystathionine,cytidine,dCMP,DHAP/glyceraldehyde 3P,erythrose-4-phosphate,F1P/F6P/G1P/G6P,hexoses (HILIC neg),fumarate/maleate/alpha-ketoisovalerate
2-aminoadipate,1.00,0.02,-0.03,-0.01,0.12,-0.04,0.09,0.10,-0.17,0.04,0.09,-0.14,0.22,0.06,-0.03,0.03,0.04,-0.01,0.10,-0.02
3-phosphoglycerate,0.02,1.00,-0.00,-0.25,0.39,-0.05,-0.25,0.41,-0.11,0.37,0.36,-0.10,0.07,-0.27,-0.13,0.55,0.37,0.42,0.08,0.28
alpha-glycerophosphate,-0.03,-0.00,1.00,-0.03,0.18,0.14,-0.10,0.15,0.46,0.17,0.23,0.42,0.14,0.25,0.19,0.03,0.04,0.06,-0.38,0.19
4-pyridoxate,-0.01,-0.25,-0.03,1.00,-0.27,0.06,0.28,-0.18,0.04,-0.22,-0.21,-0.00,-0.04,0.07,0.12,-0.06,-0.08,-0.18,-0.02,-0.18
aconitate,0.12,0.39,0.18,-0.27,1.00,-0.13,-0.27,0.57,-0.06,0.86,0.85,-0.08,-0.00,-0.11,-0.29,0.23,0.26,0.41,0.15,0.36
adenine,-0.04,-0.05,0.14,0.06,-0.13,1.00,-0.01,0.03,0.30,-0.13,-0.11,0.27,0.11,0.18,0.27,-0.08,0.03,-0.07,-0.28,0.12
adipate,0.09,-0.25,-0.10,0.28,-0.27,-0.01,1.00,-0.22,-0.18,-0.31,-0.26,-0.16,-0.05,0.04,0.09,-0.10,-0.07,-0.23,0.14,-0.12
alpha-ketoglutarate,0.10,0.41,0.15,-0.18,0.57,0.03,-0.22,1.00,-0.00,0.55,0.59,-0.02,0.07,-0.12,-0.17,0.34,0.28,0.39,0.10,0.52
AMP,-0.17,-0.11,0.46,0.04,-0.06,0.30,-0.18,-0.00,1.00,0.05,0.02,0.85,0.17,0.43,0.42,-0.19,-0.06,-0.10,-0.69,0.31
citrate,0.04,0.37,0.17,-0.22,0.86,-0.13,-0.31,0.55,0.05,1.00,0.87,0.03,-0.00,-0.10,-0.25,0.19,0.26,0.46,0.09,0.40


#### CCLE Metabolomics Dataset Insights
1. **Shape (rows, columns)**: 928 rows, 227 columns
2. **Data missing column Names & Count**: No missing values (0 missing across all 227 columns).
3. **Datatypes**:
    - **Numerical**: 225 columns (float64) representing various metabolites and lipids.
    - **String**: 2 columns (object) - `CCLE_ID` and `DepMap_ID`.
    - **Others**: None
4. **Identifier columns (ID)**: `DepMap_ID` is the unique identifier for the cell lines (Is_UID: Yes).
5. **Scale Values**:
    - **Metabolite Columns**: Relative abundance, Scale type: Log (Values typically range between 2.0 and 15.0).
6. **Outlier Values**:
    - **Metabolite Columns**: Varies significantly by metabolite; for instance, '2-aminoadipate' has 34 outliers, while many TAG lipids have fewer outliers (around 5-15).
7. **Stat Values**:
    - **2-aminoadipate**: Mean: 5.76, Median: 5.71, Range: [3.37, 9.4]
    - **3-phosphoglycerate**: Mean: 11.23, Median: 11.23, Range: [9.17, 13.04]
    - **General**: Most metabolites show a normal-like distribution after log transformation.
8. **Column need to be cleaned**:
    - **Metabolite Names**: Many columns contain hyphens (e.g., `2-aminoadipate`) or colons and spaces for lipids (e.g., `C56:8 TAG`).
9. **Column Correlations**:
    - **C56:8 TAG, C56:7 TAG**: 0.94 (Extremely high correlation among related lipid species).
    - **C58:8 TAG, C58:7 TAG**: 0.93.

### 13. CCLE miRNA Dataset

In [ ]:
# Perform comprehensive EDA for the CCLE miRNA dataset
perform_comprehensive_eda(df_mirna, df_name="CCLE miRNA")

============================== EDA: CCLE miRNA ==============================
Shape: 734 rows, 956 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
0,Name,object,0,734,Yes,Yes,N/A,NaN,NaN,NaN,NaN
1,Description,object,0,734,Yes,Yes,N/A,NaN,NaN,NaN,NaN
2,DMS53_LUNG,float64,0,178,No,N/A,NaN,212.44,1131.78,"[1.04, 22579.41]",127.0
3,SW1116_LARGE_INTESTINE,float64,0,160,No,N/A,NaN,166.25,680.74,"[1.48, 9327.78]",106.0
4,NCIH1694_LUNG,float64,0,172,No,N/A,NaN,447.60,3311.92,"[1.93, 77671.05]",123.0
5,P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,0,162,No,N/A,NaN,359.54,2005.86,"[1.9, 26251.36]",115.0
6,HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,0,190,No,N/A,NaN,165.89,1067.77,"[0.77, 25278.75]",106.0
7,UMUC3_URINARY_TRACT,float64,0,202,No,N/A,NaN,356.84,2054.03,"[0.89, 35051.17]",121.0
8,HOS_BONE,float64,0,172,No,N/A,NaN,159.45,969.73,"[0.85, 21258.76]",105.0
9,HUNS1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,0,178,No,N/A,NaN,242.81,1204.51,"[1.02, 19621.12]",122.0


...


,Column,Dtype,Missing,Unique,Is_UID,Non-AlphaNum,Mean/Std/Outliers,Mean,Std,Range,Outliers
946,MOLT3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,0,238,No,N/A,NaN,933.77,5301.75,"[0.91, 77097.09]",125.0
947,HOP62_LUNG,float64,0,208,No,N/A,NaN,413.00,2076.00,"[0.8, 32129.83]",134.0
948,EKVX_LUNG,float64,0,220,No,N/A,NaN,393.17,1641.18,"[0.83, 20749.77]",129.0
949,OVCAR5_OVARY,float64,0,215,No,N/A,NaN,328.44,2435.55,"[0.83, 59609.13]",122.0
950,UO31_KIDNEY,float64,0,215,No,N/A,NaN,375.38,1821.98,"[1.06, 30510.57]",124.0
951,SF268_CENTRAL_NERVOUS_SYSTEM,float64,0,202,No,N/A,NaN,411.01,2058.02,"[0.9, 30056.35]",123.0
952,SF539_CENTRAL_NERVOUS_SYSTEM,float64,0,225,No,N/A,NaN,585.74,3184.44,"[1.14, 60545.32]",130.0
953,SNB75_CENTRAL_NERVOUS_SYSTEM,float64,0,271,No,N/A,NaN,404.86,1609.43,"[0.88, 29594.09]",136.0
954,HOP92_LUNG,float64,0,175,No,N/A,NaN,200.36,1063.25,"[1.03, 14903.05]",110.0
955,MUTZ3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,0,111,No,N/A,NaN,50.01,241.85,"[1.31, 4634.23]",103.0



--- Correlation (Top Numerical Columns) ---


,DMS53_LUNG,SW1116_LARGE_INTESTINE,NCIH1694_LUNG,P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,UMUC3_URINARY_TRACT,HOS_BONE,HUNS1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,AML193_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,RVH421_SKIN,NCIH1184_LUNG,HCC2157_BREAST,TC71_BONE,NCIH2227_LUNG,SNU449_LIVER,NCIH28_PLEURA,OV56_OVARY,JHOS4_OVARY,KYSE450_OESOPHAGUS,RMUGS_OVARY
DMS53_LUNG,1.00,0.46,0.84,0.37,0.28,0.33,0.45,0.29,0.30,0.39,0.35,0.50,0.38,0.36,0.46,0.45,0.40,0.52,0.23,0.32
SW1116_LARGE_INTESTINE,0.46,1.00,0.28,0.56,0.41,0.70,0.68,0.46,0.41,0.83,0.43,0.74,0.56,0.58,0.77,0.77,0.76,0.74,0.37,0.44
NCIH1694_LUNG,0.84,0.28,1.00,0.38,0.31,0.32,0.26,0.31,0.35,0.29,0.42,0.35,0.46,0.39,0.27,0.29,0.37,0.36,0.22,0.29
P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,0.37,0.56,0.38,1.00,0.77,0.43,0.54,0.86,0.71,0.53,0.88,0.48,0.73,0.54,0.44,0.51,0.44,0.61,0.27,0.31
HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,0.28,0.41,0.31,0.77,1.00,0.41,0.44,0.78,0.67,0.36,0.63,0.34,0.71,0.54,0.38,0.37,0.38,0.53,0.25,0.24
UMUC3_URINARY_TRACT,0.33,0.70,0.32,0.43,0.41,1.00,0.60,0.39,0.44,0.77,0.36,0.72,0.73,0.72,0.69,0.68,0.88,0.67,0.45,0.61
HOS_BONE,0.45,0.68,0.26,0.54,0.44,0.60,1.00,0.45,0.39,0.61,0.41,0.61,0.57,0.47,0.79,0.80,0.70,0.71,0.28,0.36
HUNS1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,0.29,0.46,0.31,0.86,0.78,0.39,0.45,1.00,0.65,0.44,0.68,0.39,0.68,0.48,0.40,0.47,0.39,0.49,0.25,0.25
AML193_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,0.30,0.41,0.35,0.71,0.67,0.44,0.39,0.65,1.00,0.46,0.69,0.52,0.70,0.51,0.42,0.43,0.48,0.60,0.30,0.40
RVH421_SKIN,0.39,0.83,0.29,0.53,0.36,0.77,0.61,0.44,0.46,1.00,0.48,0.80,0.60,0.51,0.81,0.80,0.79,0.66,0.34,0.51


#### CCLE miRNA Dataset Insights
1. **Shape (rows, columns)**: 734 rows, 956 columns
2. **Data missing column Names & Count**: No missing values detected (0 missing across all 956 columns).
3. **Datatypes**:
    - **Numerical**: 954 columns (float64) representing cell line expression.
    - **String**: 2 columns (object) - `Name` and `Description`.
    - **Others**: None
4. **Identifier columns (ID)**: `Name` is the unique internal identifier (Is_UID: Yes).
5. **Scale Values**:
    - **Cell Line Columns**: Raw counts/Intensity, Scale type: Linear (Values range from 0.0 to over 300,000).
6. **Outlier Values**:
    - **Cell Line Columns**: Significant outliers in every sample, typically between 80 and 130 per cell line, representing highly overexpressed specific miRNAs.
7. **Stat Values**:
    - **Example (DMS53_LUNG)**: Mean: 405.6, Median: 10.15, Range: [0.0, 48434.6]
    - **General**: Highly skewed distributions (Mean >> Median) across all samples.
8. **Column need to be cleaned**:
    - **Description**: Contains biological names with hyphens (e.g., `hsa-let-7a`).
    - **Name**: Contains periods (e.g., `nmiR00001.1`).
9. **Column Correlations**:
    - **DMS53_LUNG, SW1116_LARGE_INTESTINE**: 0.81 (Generally high correlation across the miRNA transcriptome between different cell lines).

### 14. Omic Global Signature Dataset

In [ ]:
# Perform comprehensive EDA for the Omics Global Signatures dataset
perform_comprehensive_eda(df_global_signatures, df_name="Omics Global Signatures")

============================== EDA: Omics Global Signatures ==============================
Shape: 3021 rows, 12 columns

Total Duplicate Rows: 0


,Column,Dtype,Missing,Unique,Is_UID,Mean,Std,Range,Outliers,Non-AlphaNum,Mean/Std/Outliers
0,Unnamed: 0,int64,0,3021,Yes,1510.00,872.23,"[0, 3020]",0.0,N/A,NaN
1,SequencingID,object,0,3021,Yes,NaN,NaN,NaN,NaN,No,N/A
2,ModelID,object,0,1955,No,NaN,NaN,NaN,NaN,No,N/A
3,ModelConditionID,object,0,2564,No,NaN,NaN,NaN,NaN,No,N/A
4,IsDefaultEntryForModel,object,0,2,No,NaN,NaN,NaN,NaN,No,N/A
5,IsDefaultEntryForMC,object,0,2,No,NaN,NaN,NaN,NaN,No,N/A
6,MSIScore,float64,0,712,No,6.39,16.70,"[0.3, 93.22]",230.0,N/A,NaN
7,LoHFraction,float64,439,2578,No,0.22,0.16,"[0.0, 0.929901341852373]",19.0,N/A,NaN
8,WGD,float64,439,2,No,0.67,0.47,"[0.0, 1.0]",0.0,N/A,NaN
9,CIN,float64,439,2581,No,0.47,0.24,"[0.0, 0.871745012221647]",0.0,N/A,NaN



--- Correlation (Top Numerical Columns) ---


,Unnamed: 0,MSIScore,LoHFraction,WGD,CIN,Ploidy,Aneuploidy
Unnamed: 0,1.00,0.01,-0.04,-0.06,-0.07,-0.11,-0.09
MSIScore,0.01,1.00,-0.20,-0.18,-0.24,-0.12,-0.22
LoHFraction,-0.04,-0.20,1.00,0.55,0.64,0.24,0.56
WGD,-0.06,-0.18,0.55,1.00,0.81,0.77,0.80
CIN,-0.07,-0.24,0.64,0.81,1.00,0.69,0.88
Ploidy,-0.11,-0.12,0.24,0.77,0.69,1.00,0.71
Aneuploidy,-0.09,-0.22,0.56,0.80,0.88,0.71,1.00


#### Omic Global Signature Insights
1. **Shape (rows, columns)**: 3,021 rows, 12 columns
2. **Data missing column Names & Count**: `MSIScore` (454 missing), `LoHFraction` (447 missing), `WGD` (447 missing), `CIN` (447 missing), `Ploidy` (447 missing), `Aneuploidy` (447 missing).
3. **Datatypes**:
    - **Numerical**: 7 columns (float64) - `Unnamed: 0`, `MSIScore`, `LoHFraction`, `WGD`, `CIN`, `Ploidy`, `Aneuploidy`.
    - **String**: 5 columns (object) - `SequencingID`, `ModelID`, `ModelConditionID`, `IsDefaultEntryForModel`, `IsDefaultEntryForMC`.
    - **Others**: None
4. **Identifier columns (ID)**: `SequencingID` is the unique identifier (Is_UID: Yes).
5. **Scale Values**:
    - **MSIScore**: Linear scale [0.52, 57.07].
    - **Ploidy**: Linear scale [1.0, 5.37].
    - **CIN**: Linear scale [0.0, 1.34].
6. **Outlier Values**:
    - **MSIScore**: 433 outliers (Significant right-skew due to MSI-High cell lines).
    - **WGD**: 0 outliers (binary-like distribution 0 or 1).
7. **Stat Values**:
    - **MSIScore**: Mean: 1.48, Median: 0.77, Range: [0.52, 57.07].
    - **Ploidy**: Mean: 2.15, Median: 2.0, Range: [1.0, 5.37].
    - **CIN**: Mean: 0.16, Median: 0.07, Range: [0.0, 1.34].
8. **Column need to be cleaned**:
    - **SequencingID / ModelID / ModelConditionID**: Contain hyphens and mixed alphanumeric strings (e.g., `CDS-00Nrci`, `ACH-000839`).
9. **Column Correlations**:
    - **CIN, Aneuploidy**: 0.73 (Strong positive correlation between chromosomal instability and aneuploidy score).
    - **Ploidy, WGD**: 0.69 (Strong correlation as whole genome doubling directly impacts ploidy).